# Classical Reasoning Agent — Logical Entailment over Knowledge Bases Given in Logic

**Task.** Given a knowledge base (context) of facts and rules and a query (statement), decide whether the query is
**entailed** (`True`) or **not entailed** (`False`). The evaluation data provides the
knowledge base and the query **directly in logic**, so the system reads logic as its primary input:

```
logic text ──► logic parser ──► internal representation ──► classical reasoning ──► True / False
```

| Input recognised automatically | Internal representation | Classical reasoning |
|---|---|---|
| First-order logic (`∀x (Young(x) → Smart(x))`, ASCII `forall x. (young(x) -> smart(x))`, TPTP `![X]:`) | Formula tree → clausal normal form, or a logic program when the KB is Horn-like | **Forward chaining** + **unification** (logic programs) / **resolution refutation** with unification (general FOL) |
| Prolog-style logic programs (`smart(X) :- young(X), \+ kind(X).`) | Facts + rules with negation as failure | Forward chaining, stratified negation as failure |
| RuleTaker/ProofWriter triples (`("Bob" "is" "young" "+")`) | Facts + rules | Forward chaining, stratified negation as failure |
| CounterBench causal templates (`We know that X causes Y … Would Y occur if not X instead of X?`) | Boolean structural causal model | **Exhaustive model enumeration** (abduction) + forward propagation |

Every parser and reasoning engine is implemented below from scratch in plain Python (standard library only); no
generative model is used anywhere. `datasets` / `huggingface_hub` are used only to **download** the public data.

**How to run:** `Kernel → Restart & Run All`. The notebook downloads the public datasets, prints the accuracy and
the inference time for every dataset.

In [1]:
# ---------------------------------------------------------------- configuration
import os, sys, re, ast, json, csv, time, itertools
from collections import defaultdict, Counter

DATA_DIR = 'data'            # optional local copies of the datasets (see README.txt)
TEST_FILE = None             # e.g. 'test.csv'  -> predictions for this file are written to submission.csv
OUTPUT_CSV = 'submission.csv'
LABEL_COLUMN = 'label'       # name of the prediction column in submission.csv
SEMANTICS = 'auto'           # 'auto' | 'closed' (negation as failure) | 'open' (classical first-order entailment)
RULETAKER_SPLIT = 'test'     # tasksource/ruletaker split used for evaluation
FOLIO_SPLIT = 'validation'   # tasksource/folio split used for evaluation
MAX_PER_DATASET = None       # None = every example; an integer limits each dataset (quick runs)
PROVER_MAX_STEPS = 500       # resolution budget per query (steps)
PROVER_TIME_LIMIT = 0.5      # resolution budget per query (seconds)
sys.setrecursionlimit(10000)
print('Python', sys.version.split()[0])

Python 3.12.3


## 1. Logic parser

A tokenizer plus recursive-descent parser for first-order logic in Unicode or ASCII notation. Operator precedence,
loosest first: `↔`, `→` (right-associative, also `←`), `⊕`, `∨`, `∧`, then `¬` and quantifiers. A quantifier
followed by a bracketed group scopes over that group (`∀x (…)`), unless the variable is used again after it, in which
case the scope extends to the end of the formula. Variables are the names bound by a quantifier; free `x, y, z, u, v,
w` (and single upper-case letters) are implicitly universally quantified, as in `Young(x) → Smart(x)`. Constant and
predicate names are case-insensitive. Unbalanced brackets are repaired once before giving up.

The same module reads Prolog-style programs (`:-`, `,`, `;`, `\+`, upper-case variables) and the RuleTaker/ProofWriter
triple notation. Text that is not logic (two bare words in a row, as in *“Bob is young”*) is rejected, so English
inputs can be routed elsewhere.

In [2]:
"""Parser for knowledge bases and queries given directly in logic.

Accepted syntaxes (detected automatically):
  * First-order logic, Unicode or ASCII:  ∀x (Young(x) → Smart(x)),  forall x. (young(x) -> smart(x)),
    all x (P(x) & Q(x) => R(x)),  ![X]: (p(X) => q(X))  (TPTP),  ¬ ~ ! - not,  ∧ & && ^ and,  ∨ | || or,
    → -> => implies,  ↔ <-> <=> iff,  ⊕ xor,  = ≠ !=,  ⊤ ⊥ true false.
  * Prolog-style logic programs:  smart(X) :- young(X), \\+ kind(X).   young(bob).   (\\+ / not = negation as failure)
  * RuleTaker / ProofWriter triple representation:  ("Bob" "is" "young" "+")
    and rules  ((("something" "is" "kind" "+")) -> ("something" "is" "big" "+")).

Output: formulas in the AST used by the reasoning engines
  ('atom', pred, args) | ('not', f) | ('and', [f..]) | ('or', [f..]) | ('imp', a, b) | ('iff', a, b) |
  ('xor', a, b) | ('all', var, f) | ('ex', var, f)
Variables are strings starting with '?'; constants are lower-case strings; function terms are tuples.
"""
import ast
import re


class LogicSyntaxError(Exception):
    pass


# ------------------------------------------------------------------------------------------- tokenizer
_TOKEN = re.compile(r"""
    (?P<ws>\s+)
  | (?P<str>"[^"]*"|'[^']*')
  | (?P<op><->|<=>|->|=>|<-|:-|\?-|\\\+|!=|==|&&|\|\||:=|~=|\\=)
  | (?P<uni>[∀∃¬∧∨→↔⊕⊻⇒⇔←≠≡⊤⊥∼⋀⋁⟷⟶⟹⟺])
  | (?P<num>\d+(?:\.\d+)?(?![\w]))
  | (?P<id>[^\W\d][\w'’]*(?:\.\d[\w'’]*)*(?:-[^\W_][\w'’]*)*)
  | (?P<punct>[()\[\],.;:!~&|^=\-?{}])
""", re.X)

_CANON = {'⟶': '→', '⟹': '→', '⟷': '↔', '⟺': '↔', '⇒': '→', '->': '→', '=>': '→', 'implies': '→',
          '<->': '↔', '<=>': '↔', '⇔': '↔', '≡': '↔', 'iff': '↔',
          '⊻': '⊕', 'xor': '⊕',
          '∼': '¬', '~': '¬', 'not': '¬', '\\+': '¬',
          '&&': '∧', '&': '∧', '^': '∧', 'and': '∧', '⋀': '∧',
          '||': '∨', '|': '∨', 'or': '∨', '⋁': '∨',
          '!=': '≠', '~=': '≠', '\\=': '≠', '==': '=',
          '<-': '←', 'forall': '∀', 'exists': '∃', 'true': '⊤', 'false': '⊥'}
WORD_OPS = {'implies', 'iff', 'xor', 'not', 'and', 'or', 'forall', 'exists', 'true', 'false'}


def tokenize(text):
    tokens, pos = [], 0
    while pos < len(text):
        m = _TOKEN.match(text, pos)
        if not m:
            raise LogicSyntaxError('unexpected character %r' % text[pos])
        pos = m.end()
        kind = m.lastgroup
        tok = m.group(kind)
        if kind == 'ws':
            continue
        if kind == 'id' and tok.lower() in WORD_OPS:
            tokens.append(('op', _CANON[tok.lower()]))
        elif kind in ('op', 'uni', 'punct'):
            tokens.append(('op', _CANON.get(tok, tok)))
        else:
            tokens.append((kind, tok))
    return tokens


def _norm_name(name):
    name = name.strip().strip('"\'')
    return re.sub(r'\s+', '_', name).lower()


FREE_VAR = re.compile(r'^(?:[uvwxyz]\d*|[A-Z]\d*)$')   # free x, y, z, X, Y ... are implicitly universal


# ------------------------------------------------------------------------------------------- parser
class _Parser:
    BINARY = {'→', '↔', '⊕', '∨', '∧', '←', '=', '≠'}
    # precedence, loosest first:  ↔   →/←   ⊕   ∨   ∧   ¬/quantifiers

    def __init__(self, tokens, prolog=False):
        self.toks = tokens
        self.i = 0
        self.prolog = prolog
        self.depth = 0

    # -- helpers
    def peek(self, k=0):
        j = self.i + k
        return self.toks[j] if j < len(self.toks) else (None, None)

    def at(self, *vals):
        kind, v = self.peek()
        return kind == 'op' and v in vals

    def take(self, val=None):
        kind, v = self.peek()
        if kind is None:
            raise LogicSyntaxError('unexpected end of formula')
        if val is not None and not (kind == 'op' and v == val):
            raise LogicSyntaxError('expected %s, found %s' % (val, v))
        self.i += 1
        return kind, v

    def is_var(self, name, bound):
        if name in bound:
            return True
        if self.prolog:
            return name[0].isupper() or name[0] == '_'
        return bool(FREE_VAR.match(name))

    # -- grammar (lowest precedence first)
    def formula(self, bound):
        left = self.iff(bound)
        if self.prolog and self.at(':-'):
            self.take()
            body = self.formula(bound)
            return ('imp', body, left)
        return left

    def iff(self, bound):
        left = self.imp(bound)
        while self.at('↔'):
            self.take()
            left = ('iff', left, self.imp(bound))
        return left

    def imp(self, bound):
        left = self.xor(bound)
        if self.at('→'):
            self.take()
            return ('imp', left, self.imp(bound))
        if self.at('←'):
            self.take()
            return ('imp', self.imp(bound), left)
        return left

    def xor(self, bound):
        left = self.disj(bound)
        while self.at('⊕'):
            self.take()
            left = ('xor', left, self.disj(bound))
        return left

    def disj(self, bound):
        parts = [self.conj(bound)]
        while self.at('∨') or (self.prolog and self.at(';')):
            self.take()
            parts.append(self.conj(bound))
        return parts[0] if len(parts) == 1 else ('or', parts)

    def conj(self, bound):
        parts = [self.unary(bound)]
        while self.at('∧') or (self.at(',') and (self.prolog or self.depth > 0)):
            self.take()
            parts.append(self.unary(bound))
        return parts[0] if len(parts) == 1 else ('and', parts)

    def quantifier_vars(self):
        names = []
        if self.at('['):                     # TPTP  ![X, Y]:
            self.take('[')
            while not self.at(']'):
                kind, v = self.take()
                if kind == 'id':
                    names.append(v)
            self.take(']')
        elif self.at('(') and self.peek(1)[0] == 'id' and self.peek(2) == ('op', ')'):
            self.take('(')
            names.append(self.take()[1])     # ∀(x)
            self.take(')')
        else:
            kind, v = self.take()
            if kind != 'id':
                raise LogicSyntaxError('quantifier without variable')
            names.append(v)
            while self.at(',') and self.peek(1)[0] == 'id' and self.peek(2)[1] not in ('(',):
                self.take(',')
                names.append(self.take()[1])
        if self.at('.', ':'):
            self.take()
        return names

    def unary(self, bound):
        kind, v = self.peek()
        if kind == 'op' and v in ('¬', '-', '!') and not (v == '!' and self.peek(1) == ('op', '[')):
            self.take()
            return ('not', self.unary(bound))
        if kind == 'op' and (v in ('∀', '∃') or (v in ('!', '?') and self.peek(1) == ('op', '['))):
            self.take()
            q = 'all' if v in ('∀', '!') else 'ex'
            names = self.quantifier_vars()
            inner = bound | set(names)
            body = self._quantifier_body(names, inner)
            for n in reversed(names):
                body = (q, '?' + n, body)
            return body
        if kind == 'id' and v.lower() == 'all' and self.peek(1)[0] == 'id' and \
                self.peek(2)[1] in ('.', '(', ':', '∀', '∃', ','):
            self.take()
            names = self.quantifier_vars()
            body = self._quantifier_body(names, bound | set(names))
            for n in reversed(names):
                body = ('all', '?' + n, body)
            return body
        if kind == 'op' and v in ('(', '[', '{'):
            close = {'(': ')', '[': ']', '{': '}'}[v]
            self.take()
            self.depth += 1
            f = self.formula(bound)
            self.depth -= 1
            self.take(close)
            return self._maybe_equality_after_group(f)
        if kind == 'op' and v == '⊤':
            self.take()
            return ('atom', '$true', ())
        if kind == 'op' and v == '⊥':
            self.take()
            return ('not', ('atom', '$true', ()))
        return self.atom(bound)

    def _maybe_equality_after_group(self, f):
        return f

    def _quantifier_body(self, names, inner):
        """Tight scope for '∀x (…)' unless the variable is still used after the group (then wide scope)."""
        start = self.i
        if self.at('(', '['):
            body = self.unary(inner)
            if self.at(*self.BINARY) and self._var_used_later(names):
                self.i = start
                return self.formula(inner)
            return body
        return self.formula(inner)

    def _var_used_later(self, names):
        depth = 0
        for kind, v in self.toks[self.i:]:
            if kind == 'op' and v in ('(', '['):
                depth += 1
            elif kind == 'op' and v in (')', ']'):
                depth -= 1
                if depth < 0:
                    return False
            elif kind == 'op' and v in ('.', ';') and depth == 0:
                return False
            elif kind == 'op' and v in ('∀', '∃'):
                return False
            elif kind == 'id' and v in names:
                return True
        return False

    def term(self, bound):
        kind, v = self.take()
        if kind == 'str':
            return _norm_name(v)
        if kind == 'num':
            return v
        if kind != 'id':
            raise LogicSyntaxError('expected a term, found %s' % v)
        if self.at('('):
            self.take('(')
            args = [] if self.at(')') else self.term_list(bound)
            self.take(')')
            return (_norm_name(v),) + tuple(args)
        return '?' + v if self.is_var(v, bound) else _norm_name(v)

    def term_list(self, bound):
        args = [self.term(bound)]
        while self.at(','):
            self.take(',')
            args.append(self.term(bound))
        return args

    def atom(self, bound):
        kind, v = self.peek()
        if kind == 'str' or kind == 'num':
            left = self.term(bound)
            return self._equality(left, bound)
        if kind != 'id':
            raise LogicSyntaxError('unexpected token %s' % v)
        self.take()
        if self.at('('):
            self.take('(')
            args = [] if self.at(')') else self.term_list(bound)
            self.take(')')
            if self.at('=', '≠'):
                return self._equality((_norm_name(v),) + tuple(args), bound)
            return ('atom', _norm_name(v), tuple(args))
        if self.at('=', '≠'):
            left = '?' + v if self.is_var(v, bound) else _norm_name(v)
            return self._equality(left, bound)
        if self.is_var(v, bound) and v in bound:
            raise LogicSyntaxError('variable %s used as a formula' % v)
        return ('atom', _norm_name(v), ())          # propositional atom

    def _equality(self, left, bound):
        op = self.take()[1]
        if op not in ('=', '≠'):
            raise LogicSyntaxError('expected = or ≠')
        right = self.term(bound)
        f = ('atom', '=', (left, right))
        return ('not', f) if op == '≠' else f


# ------------------------------------------------------------------------------------------- helpers
def free_vars(f, bound=frozenset()):
    k = f[0]
    if k == 'atom':
        out = set()
        def walk(t):
            if isinstance(t, tuple):
                for a in t[1:]:
                    walk(a)
            elif isinstance(t, str) and t.startswith('?') and t not in bound:
                out.add(t)
        for a in f[2]:
            walk(a)
        return out
    if k == 'not':
        return free_vars(f[1], bound)
    if k in ('and', 'or'):
        return set().union(*[free_vars(x, bound) for x in f[1]])
    if k in ('imp', 'iff', 'xor'):
        return free_vars(f[1], bound) | free_vars(f[2], bound)
    if k in ('all', 'ex'):
        return free_vars(f[2], bound | {f[1]})
    return set()


def close_universally(f):
    for v in sorted(free_vars(f), reverse=True):
        f = ('all', v, f)
    return f


_CONTINUES = re.compile(r'(∧|∨|→|↔|⊕|⇒|⇔|&|\||->|=>|<->|<=>|:-|,|¬|~|\band|\bor|\bimplies|\biff)\s*$', re.I)


def _balanced_chunks(text):
    """Split on newlines, keeping a formula together when brackets are still open or a line ends with an
    operator (multi-line formulas and clauses)."""
    chunks, cur, depth = [], [], 0
    for line in text.splitlines():
        if not line.strip():
            continue
        cur.append(line)
        depth += line.count('(') + line.count('[') - line.count(')') - line.count(']')
        if depth <= 0 and not _CONTINUES.search(line):
            chunks.append(' '.join(cur))
            cur, depth = [], 0
    if cur:
        chunks.append(' '.join(cur))
    return chunks


def _as_list(context):
    if isinstance(context, (list, tuple)):
        return [str(c) for c in context]
    s = str(context).strip()
    if s.startswith('[') and s.endswith(']'):
        try:
            value = ast.literal_eval(s)
            if isinstance(value, (list, tuple)) and all(isinstance(x, str) for x in value):
                return list(value)
        except (ValueError, SyntaxError):
            pass
    return [s]


def detect_syntax(text):
    if re.search(r'\(\s*"[^"]+"\s+"[^"]+"\s+"[^"]+"\s+"[+-~]"\s*\)', text):
        return 'triples'
    if ':-' in text or '\\+' in text or '?-' in text:
        return 'prolog'
    return 'fol'


# ------------------------------------------------------------------------------------------- triples
TRIPLE_VARS = {'someone', 'something', 'it', 'they', 'x', 'he', 'she', 'person', 'thing'}


def _triple_literal(parts):
    subj, rel, obj, pol = parts
    s = '?x' if subj.lower() in TRIPLE_VARS else _norm_name(subj)
    if rel.lower() == 'is':
        atom = ('atom', _norm_name(obj), (s,))
    else:
        o = '?x' if obj.lower() in TRIPLE_VARS else _norm_name(obj)
        atom = ('atom', _norm_name(rel), (s, o))
    return atom if pol.strip() == '+' else ('not', atom)


def _parse_triples(text):
    formulas = []
    toks = re.findall(r'"[^"]*"|\(|\)|->', text)
    pos = 0

    def sexpr():
        nonlocal pos
        if toks[pos] != '(':
            raise LogicSyntaxError('triple syntax')
        pos += 1
        items = []
        while toks[pos] != ')':
            if toks[pos] == '(':
                items.append(sexpr())
            else:
                items.append(toks[pos].strip('"') if toks[pos] != '->' else '->')
                pos += 1
        pos += 1
        return items

    def to_formula(item):
        if len(item) == 4 and all(isinstance(x, str) for x in item):
            return _triple_literal(item)
        if '->' in item:
            k = item.index('->')
            lhs = item[:k]
            if len(lhs) == 1 and isinstance(lhs[0], list) and lhs[0] and isinstance(lhs[0][0], list):
                lhs = lhs[0]
            body = [to_formula(x) for x in lhs]
            head = to_formula(item[k + 1])
            f = ('imp', body[0] if len(body) == 1 else ('and', body), head)
            return close_universally(f)
        if len(item) == 1:
            return to_formula(item[0])
        raise LogicSyntaxError('triple syntax')

    while pos < len(toks):
        formulas.append(close_universally(to_formula(sexpr())))
    return formulas


# ------------------------------------------------------------------------------------------- entry points
_TPTP = re.compile(r'\b(?:fof|cnf|tff)\(\s*[^,()]+\s*,\s*\w+\s*,\s*(.*)\)\s*\.?\s*$')


def repair_brackets(text):
    """Drop closing brackets that have no opening partner and close brackets left open."""
    out, depth = [], 0
    for ch in text:
        if ch == '(':
            depth += 1
        elif ch == ')':
            if depth == 0:
                continue
            depth -= 1
        out.append(ch)
    return ''.join(out) + ')' * depth


def parse_formulas(text, syntax=None):
    """Parse a block of logic into a list of closed formulas (retrying once with repaired brackets)."""
    try:
        return _parse_formulas(text, syntax)
    except LogicSyntaxError:
        fixed = repair_brackets(text)
        if fixed == text:
            raise
        return _parse_formulas(fixed, syntax)


def _parse_formulas(text, syntax=None):
    syntax = syntax or detect_syntax(text)
    if syntax == 'triples':
        return _parse_triples(text)
    out = []
    for chunk in _balanced_chunks(text):
        chunk = re.sub(r'^\s*\?-', '', chunk)
        chunk = _TPTP.sub(r'\1', chunk)            # fof(name, role, formula).  ->  formula
        p = _Parser(tokenize(chunk), prolog=(syntax == 'prolog'))
        while p.peek()[0] is not None:
            if p.at('.', ';', ','):
                p.take()
                continue
            f = p.formula(frozenset())
            out.append(close_universally(f))
            kind, v = p.peek()
            if kind is None or (kind == 'op' and v in ('.', ';', ',')):
                continue
            # juxtaposed formulas are only accepted after a closing bracket, e.g. "P(a) Q(b)"
            prev = p.toks[p.i - 1]
            if prev != ('op', ')') and prev != ('op', ']'):
                raise LogicSyntaxError('juxtaposed words: not a logical formula')
    return out


def parse_logic_kb(context, skipped=None):
    """Knowledge base given in logic -> (premises, syntax). Raises LogicSyntaxError for non-logical text.
    A malformed premise is skipped (and reported in `skipped`) as long as most of the knowledge base parses."""
    pieces = _as_list(context)
    syntax = detect_syntax('\n'.join(pieces))
    if syntax == 'triples':
        return _parse_triples('\n'.join(pieces)), syntax
    premises, failed, total = [], [], 0
    for piece in pieces:
        for chunk in _balanced_chunks(piece):
            total += 1
            try:
                premises.extend(parse_formulas(chunk, syntax))
            except LogicSyntaxError:
                failed.append(chunk)
    if not premises or len(failed) * 2 > total:
        raise LogicSyntaxError('not a knowledge base in logic')
    if skipped is not None:
        skipped.extend(failed)
    return premises, syntax


def parse_logic_query(statement, syntax='fol'):
    text = str(statement).strip()
    if syntax == 'triples' and '"' not in text:
        syntax = 'fol'
    if syntax == 'fol' and detect_syntax(text) == 'triples':
        syntax = 'triples'
    q = parse_formulas(text, syntax)
    if not q:
        raise LogicSyntaxError('empty query')
    return q[0] if len(q) == 1 else ('and', q)


def parse_logic_problem(context, statement):
    """Returns (premises, query, syntax). Raises LogicSyntaxError if the input is not logic."""
    premises, syntax = parse_logic_kb(context)
    return premises, parse_logic_query(statement, syntax), syntax


def _has_atoms(f):
    return f[0] == 'atom' or any(_has_atoms(x) for x in (f[1] if f[0] in ('and', 'or') else
                                                       [f[1]] if f[0] == 'not' else
                                                       [f[1], f[2]] if f[0] in ('imp', 'iff', 'xor') else
                                                       [f[2]] if f[0] in ('all', 'ex') else []))

## 2. Forward chaining over logic programs

When every premise is a ground literal, a conjunction of literals, or an implication between conjunctions /
disjunctions of literals (`∀x (A(x) ∧ ¬B(x) → C(x))`), the knowledge base is a **normal logic program** and is evaluated
bottom-up. Rule bodies are matched against the fact base by **unification** (a backtracking join over n-ary atoms,
positive literals first); conclusions are added until a fixpoint. Two semantics are available:

* **closed world** — a negative condition / query `¬p` holds when `p` is not derivable (**negation as failure**).
  Rules are **stratified** by predicate so that `¬p` is evaluated only after every rule that can conclude `p` has
  reached its fixpoint. This is the semantics of the RuleTaker labels and of Prolog.
* **open world** — negative literals are ordinary facts that must be derived explicitly. Used as a fast, sound first
  pass before resolution.

Every derived literal remembers the rule and substitution that produced it, which gives the proof trees shown later.

In [3]:
"""Logic-program view of a knowledge base and a forward-chaining (Datalog) engine.

A knowledge base is a *normal logic program* when every formula is (after removing leading ∀) a ground literal,
a conjunction of literals, or an implication  conjunction/disjunction of literals → conjunction of literals
(biconditionals between such sides are split into two implications). Such programs are evaluated bottom-up:

  * closed world (CWA): negative body literals are tested by negation as failure, rules are stratified so that
    'not p' is only evaluated after every rule that can conclude p has reached its fixpoint;
  * open world (OWA): negative literals are ordinary facts that must be derived explicitly (sound, but without
    contrapositive reasoning — the resolution prover covers that).
"""
from collections import defaultdict


class NotAProgram(Exception):
    pass


def _is_var(t):
    return isinstance(t, str) and t.startswith('?')


def _literal(f):
    if f[0] == 'atom':
        return (True, f[1], f[2])
    if f[0] == 'not' and f[1][0] == 'atom':
        return (False, f[1][1], f[1][2])
    raise NotAProgram()


def _check_terms(args):
    for a in args:
        if isinstance(a, tuple):
            raise NotAProgram()          # function symbols: not Datalog


def _conj(f):
    if f[0] == 'and':
        out = []
        for x in f[1]:
            out.extend(_conj(x))
        return out
    lit = _literal(f)
    _check_terms(lit[2])
    if lit[1] == '=':
        raise NotAProgram()              # equality needs the equality axioms (resolution)
    return [lit]


def _dnf_bodies(f):
    """Antecedent -> list of conjunctions (a disjunctive antecedent gives one rule per disjunct)."""
    if f[0] == 'or':
        out = []
        for x in f[1]:
            out.extend(_dnf_bodies(x))
        return out
    if f[0] == 'and' and any(x[0] == 'or' for x in f[1]):
        raise NotAProgram()
    return [_conj(f)]


def to_program(formulas):
    """Formulas -> (facts, rules). facts: list of literals (pol, pred, args); rules: list of (body, head)."""
    facts, rules = [], []
    for f in formulas:
        while f[0] == 'all':
            f = f[2]
        if f[0] in ('atom', 'not', 'and'):
            lits = _conj(f)
            for lit in lits:
                if any(_is_var(a) for a in lit[2]):
                    rules.append(([], lit))
                else:
                    facts.append(lit)
        elif f[0] == 'imp':
            for body in _dnf_bodies(f[1]):
                for head in _conj(f[2]):
                    rules.append((body, head))
        elif f[0] == 'iff':
            a, b = f[1], f[2]
            for x, y in ((a, b), (b, a)):
                for body in _dnf_bodies(x):
                    for head in _conj(y):
                        rules.append((body, head))
        else:
            raise NotAProgram()
    for body, head in rules:   # every head variable must be range-restricted or ground over constants later
        pass
    return facts, rules


def query_literals(q):
    """A ground literal or conjunction of ground literals -> list of literals, else NotAProgram."""
    lits = _conj(q)
    for _, _, args in lits:
        if any(_is_var(a) for a in args):
            raise NotAProgram()
    return lits


def _subst(args, theta):
    return tuple(theta.get(a, a) if _is_var(a) else a for a in args)


def _match(pattern, fact, theta):
    th = dict(theta)
    for p, f in zip(pattern, fact):
        if _is_var(p):
            if p in th:
                if th[p] != f:
                    return None
            else:
                th[p] = f
        elif p != f:
            return None
    return th


def fmt_lit(pol, pred, args):
    s = '%s(%s)' % (pred, ', '.join(a[1:].upper() if _is_var(a) else a for a in args)) if args else pred
    return s if pol else '¬' + s


def fmt_rule(body, head):
    return (' ∧ '.join(fmt_lit(*l) for l in body) or '⊤') + ' → ' + fmt_lit(*head)


class ForwardChainer:
    def __init__(self, facts, rules, closed_world=True):
        self.cwa = closed_world
        self.rules = rules
        self.given = list(facts)
        self.facts = defaultdict(set)      # (pol, pred) -> set of argument tuples
        self.trace = {}                    # (pol, pred, args) -> (rule index, theta)
        self.firings = 0
        consts = set()
        for pol, pred, args in facts:
            self.facts[(pol, pred)].add(args)
            consts.update(args)
        for body, head in rules:
            for _, _, args in body + [head]:
                consts.update(a for a in args if not _is_var(a))
        self.constants = sorted(consts)
        self._run()

    # ---------------------------------------------------------------- queries
    def holds(self, pol, pred, args):
        if pol:
            return args in self.facts[(True, pred)]
        if self.cwa:
            return args not in self.facts[(True, pred)]
        return args in self.facts[(False, pred)]

    # ---------------------------------------------------------------- evaluation
    def _strata(self):
        preds = {h[1] for _, h in self.rules}
        level = {p: 0 for p in preds}
        n = len(preds) + 1
        for _ in range(n + 1):
            changed = False
            for body, (hp, hpred, _) in self.rules:
                for pol, pred, _ in body:
                    if pred in level:
                        need = level[pred] + (1 if not pol and self.cwa else 0)
                        if need > level[hpred]:
                            if need > n:
                                return None
                            level[hpred] = need
                            changed = True
            if not changed:
                groups = defaultdict(list)
                for i, (_, h) in enumerate(self.rules):
                    groups[level[h[1]]].append(i)
                return [groups[k] for k in sorted(groups)]
        return None

    def _solutions(self, body, theta, i=0):
        if i == len(body):
            yield theta
            return
        pol, pred, args = body[i]
        a = _subst(args, theta)
        free = [t for t in a if _is_var(t)]
        if pol or not self.cwa:
            pool = self.facts.get((pol, pred), ())
            if not free:
                if a in pool:
                    yield from self._solutions(body, theta, i + 1)
                return
            for fact in list(pool):
                th = _match(a, fact, theta)
                if th is not None:
                    yield from self._solutions(body, th, i + 1)
            return
        if not free:                                   # negation as failure on a ground atom
            if a not in self.facts.get((True, pred), ()):
                yield from self._solutions(body, theta, i + 1)
            return
        for c in self.constants:                       # unbound variable under NAF: range over the domain
            th = dict(theta)
            th[free[0]] = c
            yield from self._solutions(body, th, i)

    def _add(self, pol, pred, args, why):
        s = self.facts[(pol, pred)]
        if args in s:
            return False
        s.add(args)
        self.trace[(pol, pred, args)] = why
        return True

    def _apply(self, idxs):
        ordered = {}
        for ri in idxs:
            body, head = self.rules[ri]
            ordered[ri] = sorted(body, key=lambda l: (not (l[0] or not self.cwa),))   # positives first
        changed = True
        while changed:
            changed = False
            for ri in idxs:
                hp, hpred, hargs = self.rules[ri][1]
                for th in list(self._solutions(ordered[ri], {})):
                    self.firings += 1
                    h = _subst(hargs, th)
                    free = [t for t in h if _is_var(t)]
                    if free:                            # unsafe head variable: instantiate over the domain
                        for c in self.constants:
                            th2 = dict(th)
                            for v in free:
                                th2[v] = c
                            if self._add(hp, hpred, _subst(hargs, th2), (ri, th2)):
                                changed = True
                    elif self._add(hp, hpred, h, (ri, th)):
                        changed = True

    def _run(self):
        strata = self._strata()
        if strata is None:
            self._apply(list(range(len(self.rules))))
        else:
            for group in strata:
                self._apply(group)

    def depends_on_naf(self, pol, pred, args, seen=None):
        """True if the derivation of the literal uses a negation-as-failure step."""
        seen = set() if seen is None else seen
        key = (pol, pred, args)
        if key in seen or key not in self.trace:
            return False
        seen.add(key)
        ri, th = self.trace[key]
        body, _ = self.rules[ri]
        for bp, bpred, bargs in body:
            a = _subst(bargs, th)
            if not bp and self.cwa:
                return True
            if self.depends_on_naf(bp, bpred, a, seen):
                return True
        return False

    # ---------------------------------------------------------------- proof trees
    def explain(self, pol, pred, args, depth=0, seen=None):
        seen = set() if seen is None else seen
        pad = '  ' * depth
        name = fmt_lit(pol, pred, args)
        key = (pol, pred, args)
        if key in seen:
            return [pad + name + '  (shown above)']
        seen.add(key)
        if key in self.trace:
            ri, th = self.trace[key]
            body, head = self.rules[ri]
            lines = [pad + name + '  [rule %d: %s]' % (ri + 1, fmt_rule(body, head))]
            for bp, bpred, bargs in body:
                lines += self.explain(bp, bpred, _subst(bargs, th), depth + 1, seen)
            return lines
        if args in self.facts.get((pol, pred), ()):
            return [pad + name + '  [given fact]']
        if not pol and self.cwa:
            return [pad + name + '  [negation as failure: %s is not derivable]' % fmt_lit(True, pred, args)]
        return [pad + name + '  [not derivable]']

## 3. Resolution refutation for general first-order logic

Formulas are converted to clausal form (implication / ⊕ / ↔ elimination → negation normal form → Skolemisation →
distribution, with a size guard). To decide `KB ⊨ Q` the prover adds the clauses of `¬Q` as the **set of support** and
searches for the empty clause with binary **resolution** and factoring, using most-general **unification** with an
occurs check. The clause queue is ordered by clause length (uniform-cost / best-first search over clause size);
tautologies, duplicates (up to variable renaming) and subsumed clauses are discarded; the search is bounded by a step
and a time budget.

In [4]:
"""General first-order logic: CNF conversion, unification, and resolution refutation with set-of-support."""
import itertools
import time

# Terms: constants are plain strings, variables are strings starting with '?', Skolem/function terms are tuples
# (fname, arg1, ...). Formulas: ('atom', pred, args) | ('not', f) | ('and', [f..]) | ('or', [f..]) |
# ('imp', a, b) | ('iff', a, b) | ('xor', a, b) | ('all', var, f) | ('ex', var, f)


def Atom(pred, *args):
    return ('atom', pred, tuple(args))


def Not(f):
    return ('not', f)


def And(*fs):
    fs = [f for f in fs if f is not None]
    if not fs:
        return None
    return fs[0] if len(fs) == 1 else ('and', list(fs))


def Or(*fs):
    fs = [f for f in fs if f is not None]
    if not fs:
        return None
    return fs[0] if len(fs) == 1 else ('or', list(fs))


def is_var(t):
    return isinstance(t, str) and t.startswith('?')


def fmt_term(t):
    if isinstance(t, tuple):
        return '%s(%s)' % (t[0], ','.join(fmt_term(a) for a in t[1:]))
    return t[1:] if is_var(t) else t


def fmt_formula(f):
    k = f[0]
    if k == 'atom':
        return '%s(%s)' % (f[1], ','.join(fmt_term(a) for a in f[2]))
    if k == 'not':
        return '¬' + fmt_formula(f[1])
    if k in ('and', 'or'):
        sym = ' ∧ ' if k == 'and' else ' ∨ '
        return '(' + sym.join(fmt_formula(x) for x in f[1]) + ')'
    if k == 'imp':
        return '(' + fmt_formula(f[1]) + ' → ' + fmt_formula(f[2]) + ')'
    if k == 'iff':
        return '(' + fmt_formula(f[1]) + ' ↔ ' + fmt_formula(f[2]) + ')'
    if k == 'xor':
        return '(' + fmt_formula(f[1]) + ' ⊕ ' + fmt_formula(f[2]) + ')'
    if k == 'all':
        return '∀%s %s' % (fmt_term(f[1]), fmt_formula(f[2]))
    if k == 'ex':
        return '∃%s %s' % (fmt_term(f[1]), fmt_formula(f[2]))
    return str(f)


def fmt_clause(c):
    if not c:
        return '□ (empty clause)'
    return ' ∨ '.join(('' if s else '¬') + '%s(%s)' % (p, ','.join(fmt_term(a) for a in args))
                      for s, p, args in sorted(c, key=str))


# ---------------------------------------------------------------- CNF conversion
class CNFTooLarge(Exception):
    pass


def _elim(f):
    k = f[0]
    if k == 'atom':
        return f
    if k == 'not':
        return ('not', _elim(f[1]))
    if k in ('and', 'or'):
        return (k, [_elim(x) for x in f[1]])
    if k == 'imp':
        return ('or', [('not', _elim(f[1])), _elim(f[2])])
    if k == 'iff':
        a, b = _elim(f[1]), _elim(f[2])
        return ('and', [('or', [('not', a), b]), ('or', [a, ('not', b)])])
    if k == 'xor':
        a, b = _elim(f[1]), _elim(f[2])
        return ('and', [('or', [a, b]), ('or', [('not', a), ('not', b)])])
    if k in ('all', 'ex'):
        return (k, f[1], _elim(f[2]))
    raise ValueError(k)


def _nnf(f, neg=False):
    k = f[0]
    if k == 'atom':
        return ('not', f) if neg else f
    if k == 'not':
        return _nnf(f[1], not neg)
    if k in ('and', 'or'):
        kk = k if not neg else ('or' if k == 'and' else 'and')
        return (kk, [_nnf(x, neg) for x in f[1]])
    if k in ('all', 'ex'):
        kk = k if not neg else ('ex' if k == 'all' else 'all')
        return (kk, f[1], _nnf(f[2], neg))
    raise ValueError(k)


_counter = itertools.count()


def _subst_term(t, theta):
    if is_var(t):
        return theta.get(t, t)
    if isinstance(t, tuple):
        return (t[0],) + tuple(_subst_term(a, theta) for a in t[1:])
    return t


def _skolem(f, theta, univ):
    k = f[0]
    if k == 'atom':
        return ('atom', f[1], tuple(_subst_term(a, theta) for a in f[2]))
    if k == 'not':
        return ('not', _skolem(f[1], theta, univ))
    if k in ('and', 'or'):
        return (k, [_skolem(x, theta, univ) for x in f[1]])
    if k == 'all':
        nv = '?v%d' % next(_counter)
        th = dict(theta)
        th[f[1]] = nv
        return _skolem(f[2], th, univ + [nv])
    if k == 'ex':
        name = 'sk%d' % next(_counter)
        th = dict(theta)
        th[f[1]] = (name,) + tuple(univ) if univ else name
        return _skolem(f[2], th, univ)
    raise ValueError(k)


def _distribute(f, limit):
    k = f[0]
    if k == 'atom':
        return [[(True, f[1], f[2])]]
    if k == 'not':
        a = f[1]
        return [[(False, a[1], a[2])]]
    if k == 'and':
        out = []
        for x in f[1]:
            out.extend(_distribute(x, limit))
            if len(out) > limit:
                raise CNFTooLarge()
        return out
    if k == 'or':
        result = [[]]
        for x in f[1]:
            cx = _distribute(x, limit)
            result = [r + c for r in result for c in cx]
            if len(result) > limit:
                raise CNFTooLarge()
        return result
    raise ValueError(k)


def to_cnf(f, limit=64):
    g = _skolem(_nnf(_elim(f)), {}, [])
    clauses = []
    for c in _distribute(g, limit):
        lits = frozenset(c)
        if any((not s, p, a) in lits for s, p, a in lits):
            continue  # tautology
        clauses.append(lits)
    return clauses


# ---------------------------------------------------------------- unification
def _walk(t, theta):
    while is_var(t) and t in theta:
        t = theta[t]
    return t


def _occurs(v, t, theta):
    t = _walk(t, theta)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(_occurs(v, a, theta) for a in t[1:])
    return False


def unify(a, b, theta):
    a, b = _walk(a, theta), _walk(b, theta)
    if a == b:
        return theta
    if is_var(a):
        if _occurs(a, b, theta):
            return None
        th = dict(theta)
        th[a] = b
        return th
    if is_var(b):
        return unify(b, a, theta)
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            theta = unify(x, y, theta)
            if theta is None:
                return None
        return theta
    return None


def unify_args(xs, ys, theta):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        theta = unify(x, y, theta)
        if theta is None:
            return None
    return theta


def _resolve_term(t, theta):
    t = _walk(t, theta)
    if isinstance(t, tuple):
        return (t[0],) + tuple(_resolve_term(a, theta) for a in t[1:])
    return t


def _apply(clause, theta):
    return frozenset((s, p, tuple(_resolve_term(a, theta) for a in args)) for s, p, args in clause)


def _rename(clause, tag):
    mapping = {}

    def r(t):
        if is_var(t):
            if t not in mapping:
                mapping[t] = '?%s_%d' % (tag, len(mapping))
            return mapping[t]
        if isinstance(t, tuple):
            return (t[0],) + tuple(r(a) for a in t[1:])
        return t
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in clause)


def _canon(clause):
    """Variable-normalised representation used to detect duplicate clauses."""
    lits = sorted(clause, key=lambda l: (l[1], l[0], str(l[2])))
    mapping = {}

    def r(t):
        if is_var(t):
            if t not in mapping:
                mapping[t] = '?c%d' % len(mapping)
            return mapping[t]
        if isinstance(t, tuple):
            return (t[0],) + tuple(r(a) for a in t[1:])
        return t
    return tuple((s, p, tuple(r(a) for a in args)) for s, p, args in lits)


def _depth(t):
    if isinstance(t, tuple):
        return 1 + max([_depth(a) for a in t[1:]] or [0])
    return 0


def _subsumes(c, d):
    """True if clause c subsumes clause d (c·θ ⊆ d), small-clause backtracking check."""
    if len(c) > len(d):
        return False
    cl = list(c)

    def bt(i, theta):
        if i == len(cl):
            return True
        s, p, a = cl[i]
        for s2, p2, a2 in d:
            if s2 == s and p2 == p:
                th = _match_args(a, a2, theta)
                if th is not None and bt(i + 1, th):
                    return True
        return False
    return bt(0, {})


def _match_args(xs, ys, theta):
    """One-way matching (variables only in xs)."""
    th = dict(theta)
    for x, y in zip(xs, ys):
        if is_var(x):
            if x in th:
                if th[x] != y:
                    return None
            else:
                th[x] = y
        elif isinstance(x, tuple):
            if not isinstance(y, tuple) or x[0] != y[0] or len(x) != len(y):
                return None
            th = _match_args(x[1:], y[1:], th)
            if th is None:
                return None
        elif x != y:
            return None
    return th


class ResolutionProver:
    """Refutation prover: given-clause loop, set-of-support strategy, factoring, subsumption,
    lightest-clause-first selection (uniform-cost search over clause size)."""

    def __init__(self, max_steps=3000, max_clause_len=8, max_term_depth=3, time_limit=1.0):
        self.max_steps = max_steps
        self.max_clause_len = max_clause_len
        self.max_term_depth = max_term_depth
        self.time_limit = time_limit
        self.steps = 0

    def refute(self, kb_clauses, sos_clauses, want_proof=False):
        t0 = time.perf_counter()
        usable = []
        seen = set()
        parents = {}
        for c in kb_clauses:
            k = _canon(c)
            if k not in seen:
                seen.add(k)
                usable.append(c)
                parents[c] = ('KB',)
        queue = []
        for c in sos_clauses:
            k = _canon(c)
            if k not in seen:
                seen.add(k)
                queue.append(c)
                parents[c] = ('goal',)
        if any(len(c) == 0 for c in usable + queue):
            return True, []
        self.steps = 0
        tag = itertools.count()
        while queue:
            if self.steps > self.max_steps or time.perf_counter() - t0 > self.time_limit:
                return False, None
            queue.sort(key=len)
            given = queue.pop(0)
            if any(_subsumes(u, given) for u in usable if len(u) <= len(given)):
                continue
            usable.append(given)
            new = []
            g = _rename(given, 'g%d' % next(tag))
            for other in usable:
                o = _rename(other, 'o%d' % next(tag))
                for lit in g:
                    s, p, a = lit
                    for lit2 in o:
                        s2, p2, a2 = lit2
                        if p2 != p or s2 == s:
                            continue
                        theta = unify_args(a, a2, {})
                        if theta is None:
                            continue
                        self.steps += 1
                        res = _apply((g - {lit}) | (o - {lit2}), theta)
                        new.append((res, given, other))
                # factoring
            glits = list(g)
            for i, lit in enumerate(glits):
                for lit2 in glits[i + 1:]:
                    if lit[0] == lit2[0] and lit[1] == lit2[1]:
                        theta = unify_args(lit[2], lit2[2], {})
                        if theta is not None:
                            new.append((_apply(g, theta), given, None))
            for res, pa, pb in new:
                if len(res) == 0:
                    parents[res] = (pa, pb)
                    return True, self._proof(res, parents) if want_proof else []
                if len(res) > self.max_clause_len:
                    continue
                if any((not s, p, a) in res for s, p, a in res):
                    continue
                if any(_depth(t) > self.max_term_depth for _, _, args in res for t in args):
                    continue
                k = _canon(res)
                if k in seen:
                    continue
                seen.add(k)
                parents[res] = (pa, pb)
                queue.append(res)
        return False, None

    @staticmethod
    def _proof(empty, parents):
        lines, order, visited = [], [], set()

        def visit(c):
            if c in visited or c is None:
                return
            visited.add(c)
            par = parents.get(c, ('KB',))
            if len(par) == 2:
                visit(par[0])
                visit(par[1])
            order.append(c)
        visit(empty)
        index = {c: i + 1 for i, c in enumerate(order)}
        for c in order:
            par = parents.get(c, ('KB',))
            if len(par) == 1:
                src = 'premise' if par[0] == 'KB' else 'negated query'
            else:
                src = 'resolve(%s, %s)' % (index.get(par[0], '?'), index.get(par[1], '-') if par[1] else 'factor')
            lines.append('%2d. %-60s [%s]' % (index[c], fmt_clause(c), src))
        return lines

## 4. Logic reasoner: choosing the semantics and the algorithm

Classical first-order entailment (FOLIO) and negation as failure (RuleTaker) give different answers for the same
negative query, so the reasoner chooses explicitly (`SEMANTICS` in the configuration cell). With `'auto'`:

1. Prolog syntax, RuleTaker triples and English rule theories → **closed world**.
2. A first-order KB that is a logic program and never *concludes* a negative literal → **closed world**, unless the
   closed-world model derives an atom whose negation is stated in the KB through a negation-as-failure step (then
   the KB was written for classical reasoning, e.g. for a contrapositive argument).
3. Everything else → **open world**: forward chaining first (fast, sound), then resolution refutation.

Equality (`=`, `≠`) is handled by adding reflexivity, symmetry, transitivity and substitution axioms.

In [5]:
"""Reasoning over knowledge bases given directly in logic."""


def _uses_equality(fs):
    found = set()

    def walk(f):
        k = f[0]
        if k == 'atom':
            found.add((f[1], len(f[2])))
        elif k == 'not':
            walk(f[1])
        elif k in ('and', 'or'):
            for x in f[1]:
                walk(x)
        elif k in ('imp', 'iff', 'xor'):
            walk(f[1])
            walk(f[2])
        elif k in ('all', 'ex'):
            walk(f[2])
    for f in fs:
        walk(f)
    return found


def equality_axioms(signature):
    """Reflexivity, symmetry, transitivity and predicate substitution for '='."""
    A = lambda p, *a: ('atom', p, tuple(a))
    x, y, z = '?ea', '?eb', '?ec'
    axioms = [('all', x, A('=', x, x)),
              ('all', x, ('all', y, ('imp', A('=', x, y), A('=', y, x)))),
              ('all', x, ('all', y, ('all', z, ('imp', ('and', [A('=', x, y), A('=', y, z)]), A('=', x, z)))))]
    for pred, n in signature:
        if pred in ('=', '$true') or n == 0:
            continue
        for i in range(n):
            args = ['?ep%d' % k for k in range(n)]
            moved = list(args)
            moved[i] = '?eq'
            f = ('imp', ('and', [A('=', args[i], '?eq'), A(pred, *args)]), A(pred, *moved))
            for v in args + ['?eq']:
                f = ('all', v, f)
            axioms.append(f)
    return axioms


def _has_negative_head(rules):
    return any(not head[0] for _, head in rules)


def _has_naf_body(rules):
    return any(not pol for body, _ in rules for pol, _, _ in body)


class LogicReasoner:
    """semantics: 'auto' | 'closed' | 'open'.
    closed : stratified negation as failure (logic-programming semantics, RuleTaker labels)
    open   : classical first-order entailment (FOLIO labels)
    auto   : closed for Prolog / RuleTaker-triple / English rule-theory input, and for logic programs that never
             conclude a negative literal and whose closed-world model does not contradict a stated negative fact
             through a negation-as-failure step; open (classical) otherwise."""

    def __init__(self, semantics='auto', max_steps=500, time_limit=0.5, consistency_check=False):
        self.semantics = semantics
        self.max_steps = max_steps
        self.time_limit = time_limit
        self.consistency_check = consistency_check
        self.stats = {'forward-chaining (closed world)': 0, 'forward-chaining (open world)': 0,
                      'resolution': 0}
        self._cache = {}

    def choose_closed_world(self, syntax, program):
        """Returns True / False, or None when the closed-world fixpoint must be checked for consistency."""
        if self.semantics == 'closed':
            return program is not None
        if self.semantics == 'open' or program is None:
            return False
        if syntax in ('prolog', 'triples', 'nl-rules'):
            return True
        facts, rules = program
        if _has_negative_head(rules):
            return False
        return None if any(not pol for pol, _, _ in facts) else True

    @staticmethod
    def _closed_world_consistent(chainer, facts):
        """The closed-world model must not derive an atom whose negation is stated in the knowledge base."""
        return not any(not pol and chainer.holds(True, pred, args) and chainer.depends_on_naf(True, pred, args)
                       for pol, pred, args in facts)

    def decide(self, premises, query, syntax='fol', trace=None, cache_key=None):
        try:
            program = to_program(premises)
            qlits = query_literals(query)
        except NotAProgram:
            program, qlits = None, None
        closed = self.choose_closed_world(syntax, program)
        if program is not None:
            key = (cache_key, closed)
            chainer = self._cache.get(key) if cache_key is not None else None
            if chainer is None:
                chainer = ForwardChainer(program[0], program[1], closed_world=closed is not False)
                if closed is None:
                    closed = self._closed_world_consistent(chainer, program[0])
                    if not closed:
                        chainer = ForwardChainer(program[0], program[1], closed_world=False)
                if cache_key is not None:
                    if len(self._cache) > 256:
                        self._cache.clear()
                    self._cache[key] = chainer
            closed = chainer.cwa
            ok = all(chainer.holds(*l) for l in qlits)
            if closed or ok:
                self.stats['forward-chaining (%s world)' % ('closed' if closed else 'open')] += 1
                if trace is not None:
                    self._fc_trace(trace, chainer, program, qlits, ok, closed)
                return ok
        return self._resolution(premises, query, trace)

    # ---------------------------------------------------------------- resolution
    def _resolution(self, premises, query, trace):
        self.stats['resolution'] += 1
        kb = []
        sig = _uses_equality(premises + [query])
        extra = equality_axioms(sig) if any(p == '=' for p, _ in sig) else []
        for f in premises + extra:
            try:
                kb.extend(to_cnf(f))
            except CNFTooLarge:
                pass
        if any(p == '$true' for p, _ in sig):
            kb.append(frozenset([(True, '$true', ())]))
        try:
            goal = to_cnf(('not', query))
        except CNFTooLarge:
            return False
        prover = ResolutionProver(self.max_steps, time_limit=self.time_limit)
        proved, proof = prover.refute(kb, goal, want_proof=trace is not None)
        consistent = True
        if proved and self.consistency_check:
            other = ResolutionProver(self.max_steps, time_limit=self.time_limit)
            consistent = not other.refute(kb, to_cnf(query))[0]
        if trace is not None:
            trace.update(method='resolution refutation (set of support, uniform-cost clause selection)',
                         premises=[fmt_formula(f) for f in premises], query=fmt_formula(query),
                         clauses=len(kb), proof=proof if proved else None, resolution_steps=prover.steps)
        return proved and consistent

    @staticmethod
    def _fc_trace(trace, chainer, program, qlits, ok, closed):
        proof = []
        for pol, pred, args in qlits:
            if chainer.holds(pol, pred, args):
                proof += chainer.explain(pol, pred, args)
            elif not pol and closed:
                proof.append('%s fails: %s is derivable' % (fmt_lit(pol, pred, args), fmt_lit(True, pred, args)))
                proof += chainer.explain(True, pred, args, 1)
            else:
                proof.append('%s is not derivable' % fmt_lit(pol, pred, args))
        trace.update(method='forward chaining (%s world%s)' % ('closed' if closed else 'open',
                                                              ', stratified negation as failure' if closed else ''),
                     facts=[fmt_lit(*f) for f in program[0]], rules=[fmt_rule(b, h) for b, h in program[1]],
                     query=' ∧ '.join(fmt_lit(*l) for l in qlits), proof=proof, rule_firings=chainer.firings)

## 5. Causal / counterfactual stories: structural causal model + enumeration

CounterBench items are written in a fixed formal template (*“We know that Ziklo causes Blaf, Blaf or Trune causes
Vork … We observed Trune. Would Lumbo occur if not Ziklo instead of Ziklo?”*). Each relation becomes a Boolean
structural equation (`X causes Y` → `Y := X`, `X causes not Y` → `Y := ¬X`, `X and Y together cause Z` → `Z := X ∧ Y`,
`X or not Y causes Z` → `Z := X ∨ ¬Y`); variables without an equation are exogenous (prior `Bern(p)` when given,
otherwise 0.5). Observations and the factual value implied by *“instead of X”* are evidence; *“if not X”*, joint and
nested conditions are interventions. Counterfactuals follow Pearl's three steps, carried out by classical search:
**abduction** (enumerate every assignment of the exogenous variables, propagate the equations forward in topological
order, keep the assignments consistent with the evidence), **action** (replace the equations of the intervened
variables) and **prediction** (propagate again and read the target).

In [6]:
"""CounterBench-style deterministic counterfactual reasoning.
Natural language -> boolean structural causal model -> abduction / action / prediction by model enumeration."""
import itertools
import re


class CausalParseError(Exception):
    pass


def _lit(tok):
    tok = tok.strip()
    m = re.match(r'^(not\s+)?([A-Za-z][\w-]*)$', tok)
    if not m:
        raise CausalParseError('literal: ' + tok)
    return (m.group(1) is None, m.group(2))


def _expr(text):
    """'A or not B' / 'A and B' / 'A' -> ('or'|'and', [(pol, var), ...])"""
    text = text.strip()
    if re.search(r'\bor\b', text):
        return ('or', [_lit(t) for t in re.split(r'\s*,?\s*\bor\b\s*|\s*,\s*', text) if t.strip()])
    parts = [t for t in re.split(r'\s*,?\s*\band\b\s*|\s*,\s*', text) if t.strip()]
    return ('and', [_lit(t) for t in parts])


def is_causal_text(context, statement):
    c = ' '.join(context) if isinstance(context, (list, tuple)) else str(context)
    return bool(re.search(r'\bcauses?\b', c)) and bool(
        re.search(r'\boccur|\bWould\b|\bAssume\b|\bsuppose\b', str(statement), flags=re.I))


def parse_causal(context, statement):
    text = ' '.join(context) if isinstance(context, (list, tuple)) else str(context)
    full = text + ' ' + str(statement)
    priors = {}
    for v, p in re.findall(r'([A-Za-z][\w-]*)\s*~\s*Bern\(\s*([0-9.]+)\s*\)', full):
        priors[v] = float(p)
    full_wo_priors = re.sub(r'[A-Za-z][\w-]*\s*~\s*Bern\(\s*[0-9.]+\s*\)\.?', ' ', full)
    equations = {}   # var -> list of (op, lits, effect_polarity); combined with OR
    observations = {}
    know = re.findall(r'We know that (.*?)(?:\.(?:\s|$)|$)', full_wo_priors)
    for block in know:
        pieces = [p.strip() for p in re.split(r',\s*', block) if p.strip()]
        last = None
        for piece in pieces:
            piece = re.sub(r'^and\s+', '', piece)
            m = re.match(r'^(.*?)\s+(?:together\s+)?(?:causes|cause)\s+(.*)$', piece)
            if m:
                cause = _expr(m.group(1))
                effects = [e for e in re.split(r'\s+and\s+', m.group(2)) if e.strip()]
                for e in effects:
                    pol, var = _lit(e)
                    equations.setdefault(var, []).append((cause, pol))
                last = cause
            elif last is not None:  # continuation of an effect list: "X causes A, B and C"
                for e in re.split(r'\s+and\s+', piece):
                    pol, var = _lit(e)
                    equations.setdefault(var, []).append((last, pol))
            else:
                raise CausalParseError('piece: ' + piece)
    if not equations:
        raise CausalParseError('no causal relations')
    for obs in re.findall(r'We observed (.*?)(?:\.(?:\s|$)|$)', full_wo_priors):
        obs = re.sub(r'\s+occurs?\b', '', obs)
        for part in re.split(r'\s*,\s*|\s+and\s+', obs):
            if part.strip():
                pol, var = _lit(part)
                observations[var] = pol
    q = str(statement)
    interventions = []
    factual = {}
    target = None
    m = re.search(r'Would\s+(not\s+)?([A-Za-z][\w-]*)\s+(?:not\s+)?occur\s+if\s+(.*?)\?', q)
    if m:
        target = (m.group(1) is None, m.group(2))
        cond = m.group(3)
        for part in re.split(r'\s*,\s*|\s+and\s+', cond):
            part = part.strip()
            if not part:
                continue
            mi = re.match(r'^(not\s+)?([A-Za-z][\w-]*)\s+instead of\s+(not\s+)?([A-Za-z][\w-]*)$', part)
            if mi:   # "not X instead of X": intervention X = False, factual X = True
                interventions.append((mi.group(2), mi.group(1) is None))
                factual[mi.group(4)] = mi.group(3) is None
            else:
                pol, var = _lit(part)
                interventions.append((var, pol))
    else:
        for a in re.findall(r'(?:Assume|suppose)\s+(not\s+)?([A-Za-z][\w-]*)', q, flags=re.I):
            interventions.append((a[1], a[0] == ''))
        m2 = re.search(r'Would\s+(not\s+)?([A-Za-z][\w-]*)\s+occur', q)
        if not m2:
            raise CausalParseError('question')
        target = (m2.group(1) is None, m2.group(2))
    if target is None or not interventions:
        raise CausalParseError('question')
    return equations, priors, observations, interventions, factual, target


def _eval(expr, values):
    op, lits = expr
    vals = [values[v] == pol for pol, v in lits]
    return all(vals) if op == 'and' else any(vals)


class CausalModel:
    def __init__(self, equations, priors):
        self.equations = equations
        self.priors = priors
        variables = set(equations)
        for eqs in equations.values():
            for (op, lits), _ in eqs:
                variables.update(v for _, v in lits)
        self.variables = variables
        self.exogenous = sorted(v for v in variables if v not in equations)
        self.order = self._topological()

    def _topological(self):
        order, state = [], {}

        def visit(v):
            if state.get(v) == 1:
                raise CausalParseError('cycle')
            if state.get(v) == 2:
                return
            state[v] = 1
            for (op, lits), _ in self.equations.get(v, []):
                for _, p in lits:
                    visit(p)
            state[v] = 2
            order.append(v)

        for v in sorted(self.variables):
            visit(v)
        return order

    def propagate(self, exo, do=None):
        """Forward propagation through the structural equations (forward chaining)."""
        do = do or {}
        values = {}
        for v in self.order:
            if v in do:
                values[v] = do[v]
            elif v in self.equations:
                values[v] = any(_eval(expr, values) == pol for expr, pol in self.equations[v])
            else:
                values[v] = exo[v]
        return values

    def counterfactual(self, evidence, do, target):
        """Abduction (enumerate exogenous worlds consistent with evidence), action (do), prediction."""
        num = den = 0.0
        worlds = 0
        for bits in itertools.product((False, True), repeat=len(self.exogenous)):
            exo = dict(zip(self.exogenous, bits))
            w = 1.0
            for v, b in exo.items():
                p = self.priors.get(v, 0.5)
                w *= p if b else (1 - p)
            fact = self.propagate(exo)
            if any(fact[v] != val for v, val in evidence.items() if v in fact):
                continue
            worlds += 1
            cf = self.propagate(exo, do)
            den += w
            if cf[target[1]] == target[0]:
                num += w
        return (num / den if den > 0 else None), worlds


def fmt_equations(equations):
    out = []
    for var in sorted(equations):
        terms = []
        for (op, lits), pol in equations[var]:
            sym = ' ∧ ' if op == 'and' else ' ∨ '
            e = sym.join(('' if p else '¬') + v for p, v in lits)
            if len(lits) > 1:
                e = '(' + e + ')'
            terms.append(e if pol else '¬' + e)
        out.append('%s := %s' % (var, ' ∨ '.join(terms)))
    return out


def causal_answer(context, statement, trace=None):
    equations, priors, observations, interventions, factual, target = parse_causal(context, statement)
    model = CausalModel(equations, priors)
    do = {v: val for v, val in interventions}
    evidence_sets = [dict(observations, **factual), dict(observations), {}]
    prob, worlds = None, 0
    used = None
    for ev in evidence_sets:
        prob, worlds = model.counterfactual(ev, do, target)
        if prob is not None:
            used = ev
            break
    if trace is not None:
        trace.update(equations=fmt_equations(equations), priors=priors, evidence=used, do=do, target=target,
                     worlds=worlds, probability=prob, order=model.order, exogenous=model.exogenous)
    return prob is not None and prob > 0.5

## 6. Translating the English RuleTaker data into logic (data preparation)

The Hugging Face release of RuleTaker contains English only, while the test data comes in logic. To evaluate the
logic pipeline on RuleTaker, each public theory is translated once into first-order logic by a rule-based parser for
the RuleTaker sentence templates (*If … then …*, *All red, big people are kind*, relations, *can / cannot*, negation),
with a lexical fallback for paraphrased theories. The agent also uses this translator if it is ever given an English
rule theory.

In [7]:
"""Parser for RuleTaker-style English rule theories (used to turn the public English RuleTaker data into logic)."""
import re
from collections import defaultdict

VAR = '?x'
# --------------------------------------------------------------------------------------
# Text utilities
# --------------------------------------------------------------------------------------
_ABBR = re.compile(r'\b(?:Mr|Mrs|Ms|Dr|St|Jr|Sr|Inc|Ltd|Co|vs|No|Prof|Mt|Ft|Gen|Capt|Lt)\.')


def split_sentences(text):
    """Split a context (string or list of strings) into sentences."""
    if isinstance(text, (list, tuple)):
        chunks = [str(t) for t in text]
    else:
        chunks = re.split(r'\n+', str(text))
    out = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue
        protected = _ABBR.sub(lambda m: m.group(0).replace('.', '<DOT>'), chunk)
        protected = re.sub(r'\b((?:[A-Z]\.){2,})', lambda m: m.group(0).replace('.', '<DOT>'), protected)
        protected = re.sub(r'(\d)\.(\d)', r'\1<DOT>\2', protected)
        for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"\'(])', protected):
            s = s.replace('<DOT>', '.').strip()
            if s:
                out.append(s)
    return out


def verb_lemma(w):
    w = w.lower()
    irregular = {'has': 'have', 'does': 'do', 'is': 'be', 'are': 'be', 'goes': 'go'}
    if w in irregular:
        return irregular[w]
    if w.endswith('ies') and len(w) > 4:
        return w[:-3] + 'y'
    if w.endswith(('ches', 'shes', 'sses', 'xes', 'zes')):
        return w[:-2]
    if w.endswith('s') and not w.endswith('ss') and len(w) > 2:
        return w[:-1]
    return w


# --------------------------------------------------------------------------------------
# Parser
# --------------------------------------------------------------------------------------
VAR_SUBJECTS = {
    'someone', 'something', 'it', 'they', 'he', 'she', 'a person', 'a thing', 'the person', 'the thing',
    'that person', 'that thing', 'people', 'things', 'anyone', 'anything', 'everyone', 'everything',
    'somebody', 'anybody', 'everybody', 'a', 'an', 'x', 'them', 'one', 'someone who', 'something that'
}
AUX = {'is', 'are', 'was', 'were', 'does', 'do', 'did', 'can', 'cannot', "can't", 'has', 'have', 'not',
       'will', "doesn't", "don't", "isn't", "aren't"}
GENERIC_NOUNS = {'people', 'things', 'persons', 'animals', 'person', 'thing', 'animal', 'ones', 'someone', 'something'}
STOP_ATTR = {'and', 'or', 'if', 'then', 'the', 'not'}


class ParseError(Exception):
    pass


def _entity_span(tokens, i):
    """Return (entity_string, next_index) for an entity starting at tokens[i], or None."""
    if i >= len(tokens):
        return None
    t = tokens[i]
    tl = t.lower()
    if tl == 'the':
        j = i + 1
        words = []
        while j < len(tokens) and len(words) < 3:
            w = tokens[j].lower()
            if w in AUX or w in STOP_ATTR:
                break
            if words and w.endswith('s') and not w.endswith('ss') and (
                    j + 1 >= len(tokens) or tokens[j + 1].lower() in ('the',) or tokens[j + 1][:1].isupper()
                    or tokens[j + 1].lower() in VAR_SUBJECTS):
                break  # verb such as "eats the dog"
            words.append(w)
            j += 1
        if not words:
            return None
        return ' '.join(words), j
    if t[:1].isupper() and tl not in ('if', 'all', 'the', 'it', 'they', 'someone', 'something', 'people', 'things'):
        return tl, i + 1
    return None


def _var_span(tokens, i):
    for n in (3, 2, 1):
        cand = ' '.join(tokens[i:i + n]).lower()
        if len(tokens[i:i + n]) == n and cand in VAR_SUBJECTS:
            return VAR, i + n
    return None


def _subject(tokens, i):
    v = _var_span(tokens, i)
    if v:
        return v
    return _entity_span(tokens, i)


def _object(tokens, i):
    """Object of a relation: entity or variable word."""
    v = _var_span(tokens, i)
    if v:
        return v
    e = _entity_span(tokens, i)
    if e:
        return e
    return None


def parse_predicate(subj, toks):
    """Parse a predicate (token list, subject removed). Returns list of (polarity, atom) literals."""
    if not toks:
        raise ParseError('empty predicate')
    low = [t.lower() for t in toks]
    # it is not true that ...
    if low[:4] == ['it', 'is', 'not', 'true'] or low[:3] == ['is', 'not', 'true']:
        raise ParseError('nested negation')
    head = low[0]
    if head in ('is', 'are', 'was', 'were', "isn't", "aren't"):
        rest = low[1:]
        pol = True
        if head in ("isn't", "aren't"):
            pol = False
        if rest and rest[0] == 'not':
            pol = False
            rest = rest[1:]
        if rest and rest[0] in ('a', 'an'):
            rest = rest[1:]
        if not rest:
            raise ParseError('no attribute')
        # "is big and red" handled by caller; here a single attribute (1-2 words)
        if len(rest) > 1 or any(r in STOP_ATTR for r in rest):
            raise ParseError('complex attribute: ' + ' '.join(rest))
        return [(pol, ('is', subj, ' '.join(rest)))]
    if head in ('cannot', "can't") or low[:2] == ['can', 'not']:
        rest = low[2:] if head == 'can' else low[1:]
        if len(rest) != 1:
            raise ParseError('can-not form')
        return [(False, ('can', subj, rest[0]))]
    if head == 'can':
        if len(low) != 2:
            raise ParseError('can form')
        return [(True, ('can', subj, low[1]))]
    pol = True
    vtoks = toks
    if head in ('does', 'do', "doesn't", "don't"):
        if head in ("doesn't", "don't"):
            vtoks = toks[1:]
        elif len(low) > 1 and low[1] == 'not':
            vtoks = toks[2:]
        else:
            raise ParseError('do-form')
        pol = False
        if not vtoks:
            raise ParseError('missing verb')
        verb = vtoks[0].lower()
    else:
        verb = verb_lemma(vtoks[0])
    if len(vtoks) == 1:
        return [(pol, ('do', subj, verb))]
    obj = _object(vtoks, 1)
    if not obj or obj[1] != len(vtoks):
        raise ParseError('bad object: ' + ' '.join(vtoks))
    return [(pol, (verb, subj, obj[0]))]


def parse_clause(tokens, prev=None):
    """Parse one clause such as 'the cat is red' / 'it chases the dog' / 'not red' (elliptic).
    prev = (subj, verb_head_tokens) of the previous clause for ellipsis."""
    s = _subject(tokens, 0)
    if s and s[1] < len(tokens):
        subj, j = s
        pred = tokens[j:]
        lits = parse_predicate(subj, pred)
        return lits, (subj, pred[:1] if pred[0].lower() in ('is', 'are', 'was', 'were') else None)
    if prev is None:
        raise ParseError('no subject')
    subj, cop = prev
    if cop is None:
        raise ParseError('ellipsis without copula')
    lits = parse_predicate(subj, cop + tokens)
    return lits, prev


def parse_conjunction(text, prev=None):
    pieces = re.split(r'\s+and\s+|,\s*', text.strip())
    lits = []
    for p in pieces:
        p = p.strip()
        if not p:
            continue
        toks = p.split()
        l, prev = parse_clause(toks, prev)
        lits.extend(l)
    return lits, prev


def _norm(sent):
    s = sent.strip().rstrip('.!?').strip()
    s = re.sub(r'\s+', ' ', s)
    return s


def parse_rt_sentence(sent):
    """Returns ('fact', [lits]) or ('rule', [(body, head)])."""
    s = _norm(sent)
    low = s.lower()
    m = re.match(r'^if (.+?),? then (.+)$', s, flags=re.I)
    if m:
        body, prev = parse_conjunction(m.group(1))
        heads, _ = parse_conjunction(m.group(2), prev)
        return 'rule', [(body, h) for h in heads]
    # "All red, big people are kind." / "Red, cold things are round." / "Big people are not red."
    m = re.match(r'^(?:all )?([a-z ,]+?) (people|things|persons|animals|ones) (are|is) (not )?(?:an? )?([a-z]+)$', low)
    if m:
        attrs = [a for a in re.split(r'[ ,]+', m.group(1)) if a and a != 'and']
        if not attrs:
            raise ParseError('no attrs')
        body = [(True, ('is', VAR, a)) for a in attrs]
        head = (m.group(4) is None, ('is', VAR, m.group(5)))
        return 'rule', [(body, head)]
    # Fact
    toks = s.split()
    lits, _ = parse_conjunction(s)
    for pol, atom in lits:
        if VAR in atom:
            raise ParseError('variable in fact')
    return 'fact', lits


def parse_rt_statement(statement):
    s = _norm(statement)
    lits, _ = parse_conjunction(s)
    for pol, atom in lits:
        if VAR in atom:
            raise ParseError('variable in query')
    return lits


class RTTheory:
    def __init__(self, facts, rules):
        self.facts = facts          # list of (pol, atom)
        self.rules = rules          # list of (body, head)


def parse_rt_theory(context):
    facts, rules, failures = [], [], []
    for sent in split_sentences(context):
        try:
            kind, payload = parse_rt_sentence(sent)
        except ParseError as e:
            failures.append(sent)
            continue
        if kind == 'fact':
            facts.extend(payload)
        else:
            rules.extend(payload)
    return RTTheory(facts, rules), failures


# --------------------------------------------------------------------------------------
# Fallback lexical parser for paraphrased RuleTaker theories (e.g. depth-3ext-NatLang)
# --------------------------------------------------------------------------------------
RT_ATTRIBUTES = {'big', 'blue', 'cold', 'furry', 'green', 'kind', 'nice', 'quiet', 'red', 'rough', 'round',
                 'smart', 'white', 'young'}
NEG_WORDS = {'not', "n't", 'never', 'no', "isn't", "aren't", "doesn't", "don't", 'nor'}
FUNCTION_CAPS = {'if', 'all', 'the', 'a', 'an', 'any', 'some', 'every', 'each', 'someone', 'something', 'people',
                 'things', 'in', 'when', 'those', 'this', 'that', 'they', 'he', 'she', 'it', 'his', 'her',
                 'young', 'big', 'red', 'blue', 'green', 'kind', 'nice', 'cold', 'round', 'rough', 'white',
                 'smart', 'quiet', 'furry', 'people', 'anyone', 'everyone', 'somebody', 'if', 'although',
                 'because', 'since', 'while', 'after', 'before', 'once', 'whenever', 'everybody', 'other',
                 'most', 'many', 'being', 'usually', 'often', 'sometimes', 'whoever', 'folks', 'individuals',
                 'there', 'then', 'what', 'who', 'its', 'their', 'even', 'though', 'also', 'nobody', 'no'}
PRONOUNS = {'he', 'she', 'his', 'her', 'him', 'they', 'their', 'them', 'it', 'its'}


def _words(sent):
    return re.findall(r"[A-Za-z]+(?:'[a-z]+)?|n't", sent)


def _attr_lits(words, vocab):
    lits = []
    low = [w.lower() for w in words]
    for i, w in enumerate(low):
        if w in vocab:
            window = low[max(0, i - 3):i]
            pol = not any(x in NEG_WORDS or x.endswith("n't") for x in window)
            lits.append((pol, w))
    return lits


def parse_natlang_theory(context, vocab=None, names=None):
    """Lexical fallback: every sentence is a fact about a named entity (all attribute words it
    contains) or a rule (all attribute words but the last form the body, the last one is the head)."""
    vocab = set(vocab or ()) | RT_ATTRIBUTES
    sents = split_sentences(context)
    if names is None:
        names = set()
        for s in sents:
            for i, w in enumerate(_words(s)):
                if w[:1].isupper() and w.lower() not in FUNCTION_CAPS:
                    names.add(w.lower())
    facts, rules, failures = [], [], []
    prev_entity = None
    for s in sents:
        try:
            kind, payload = parse_rt_sentence(s)
            if kind == 'fact':
                facts.extend(payload)
                prev_entity = payload[0][1][1]
            else:
                rules.extend(payload)
            continue
        except ParseError:
            pass
        words = _words(s)
        low = [w.lower() for w in words]
        ents = [w for w in low if w in names]
        lits = _attr_lits(words, vocab)
        if not lits:
            failures.append(s)
            continue
        if ents:
            e = ents[0]
            facts.extend((p, ('is', e, a)) for p, a in lits)
            prev_entity = e
        elif len(lits) >= 2:
            body = [(p, ('is', VAR, a)) for p, a in lits[:-1]]
            if not any(p for p, _ in body):
                failures.append(s)
                continue
            rules.append((body, (lits[-1][0], ('is', VAR, lits[-1][1]))))
        elif prev_entity is not None and any(w in PRONOUNS for w in low):
            facts.extend((p, ('is', prev_entity, a)) for p, a in lits)
        else:
            failures.append(s)
    return RTTheory(facts, rules), failures, names


def parse_natlang_statement(statement, vocab, names):
    try:
        return parse_rt_statement(statement)
    except ParseError:
        pass
    words = _words(statement)
    low = [w.lower() for w in words]
    ents = [w for w in low if w in names]
    lits = _attr_lits(words, set(vocab) | RT_ATTRIBUTES)
    if not ents or not lits:
        raise ParseError('statement')
    return [(p, ('is', ents[0], a)) for p, a in lits]


"""Serialise a parsed rule theory (rt_engine representation) as first-order logic text."""


def _name(s):
    return 'x' if s == VAR else s.replace(' ', '_')


def _pred(word):
    return ''.join(p.capitalize() for p in word.replace('-', ' ').split())


def literal_to_fol(pol, atom):
    rel, a, b = atom
    if rel == 'is':
        s = '%s(%s)' % (_pred(b), _name(a))
    elif rel in ('can', 'do'):
        s = '%s%s(%s)' % (rel.capitalize(), _pred(b), _name(a))
    else:
        s = '%s(%s, %s)' % (_pred(rel), _name(a), _name(b))
    return s if pol else '¬' + s


def theory_to_fol(context):
    """English rule theory -> (list of FOL strings, entity names) or (None, None) if it cannot be parsed.
    Template sentences use the RuleTaker grammar; paraphrased theories use the lexical fallback."""
    theory, failures = parse_rt_theory(context)
    names = None
    if failures:
        theory, failures, names = parse_natlang_theory(context)
        if failures:
            return None, None
    out = [literal_to_fol(p, a) for p, a in theory.facts]
    for body, head in theory.rules:
        b = ' ∧ '.join(literal_to_fol(p, a) for p, a in body)
        h = literal_to_fol(*head)
        uses_var = any(VAR in a for _, a in body + [head])
        out.append(('∀x (%s → %s)' if uses_var else '(%s → %s)') % (b, h))
    return out, names


def statement_to_fol(statement, names=None):
    try:
        lits = parse_rt_statement(statement)
    except ParseError:
        if names is None:
            raise
        lits = parse_natlang_statement(statement, set(), names)
    return ' ∧ '.join(literal_to_fol(p, a) for p, a in lits)

## 7. The reasoning agent

In [8]:
"""Top-level agent: logic input -> representation -> classical reasoning -> True / False."""


class ReasoningAgent:
    """Routes each (context, statement) pair:
         1. causal / counterfactual story  -> structural causal model, enumeration + propagation
         2. knowledge base given in logic  -> forward chaining (logic programs) or resolution (general FOL)
         3. plain-English rule theory      -> converted to logic by the rule-theory parser, then as in 2."""

    def __init__(self, semantics='auto', max_steps=500, time_limit=0.5):
        self.logic = LogicReasoner(semantics, max_steps=max_steps, time_limit=time_limit)
        self._kb_cache = {}
        self.stats = {'causal model': 0, 'logic input': 0, 'English rule theory -> logic': 0, 'unparsed': 0}

    @staticmethod
    def _key(context):
        return context if isinstance(context, str) else '\n'.join(map(str, context))

    def to_logic(self, context, statement):
        """Returns (premises, query, syntax, origin)."""
        key = self._key(context)
        cached = self._kb_cache.get(key)
        if cached is None:
            try:
                premises, syntax = parse_logic_kb(context)
                cached = ('logic', premises, syntax, None)
            except LogicSyntaxError:
                fol, names = theory_to_fol(context)
                cached = ('nl', parse_formulas('\n'.join(fol), 'fol'), 'nl-rules', names) if fol else \
                    ('none', None, None, None)
            if len(self._kb_cache) > 2000:
                self._kb_cache.clear()
            self._kb_cache[key] = cached
        origin, premises, syntax, names = cached
        if origin == 'logic':
            query = parse_logic_query(statement, syntax)
        elif origin == 'nl':
            query = parse_logic_query(statement_to_fol(statement, names), 'fol')
        else:
            raise LogicSyntaxError('context is neither logic nor a parsable rule theory')
        return premises, query, syntax, origin

    def predict(self, context, statement, trace=None):
        if is_causal_text(context, statement):
            try:
                info = {} if trace is not None else None
                ans = causal_answer(context, statement, info)
                self.stats['causal model'] += 1
                if trace is not None:
                    trace.update(method='structural causal model (abduction, action, prediction by enumeration)',
                                 **info)
                return ans
            except CausalParseError:
                pass
        try:
            premises, query, syntax, origin = self.to_logic(context, statement)
        except (LogicSyntaxError, ParseError):
            self.stats['unparsed'] += 1
            if trace is not None:
                trace.update(method='input could not be parsed', note='answer False')
            return False
        self.stats['logic input' if origin == 'logic' else 'English rule theory -> logic'] += 1
        if trace is not None:
            trace['input syntax'] = syntax
        return self.logic.decide(premises, query, syntax, trace, cache_key=self._key(context))

## 8. Demonstration: the assignment example in several logic notations (with reasoning traces)

In [9]:
def show_trace(context, statement, agent=None, label=None):
    agent = agent or ReasoningAgent(SEMANTICS, PROVER_MAX_STEPS, PROVER_TIME_LIMIT)
    trace = {}
    answer = agent.predict(context, statement, trace)
    ctx = context if isinstance(context, str) else '\n'.join(map(str, context))
    print('KNOWLEDGE BASE:')
    for line in ctx.splitlines():
        print('   ', line)
    print('QUERY   :', statement)
    for key in ('input syntax', 'method', 'facts', 'rules', 'premises', 'query', 'equations', 'priors', 'evidence',
                'do', 'target', 'worlds', 'probability'):
        value = trace.get(key)
        if value in (None, [], {}):
            continue
        if isinstance(value, list) and key in ('facts', 'rules', 'premises', 'equations'):
            print('%-8s:' % key.upper())
            for v in value:
                print('    ', v)
        else:
            print('%-8s: %s' % (key.upper(), value))
    if trace.get('proof'):
        print('PROOF   :')
        for line in trace['proof']:
            print('    ', line)
    print('ANSWER  :', answer, '' if label is None else '  (gold: %s)' % label)
    print('-' * 100)
    return answer

# the example from the assignment, in first-order logic
show_trace(['Young(Bob)', 'Kind(Bob)', '∀x (Young(x) → Smart(x))', '∀x (Smart(x) ∧ Kind(x) → Successful(x))'],
           'Successful(Bob)')
# the same knowledge base in ASCII notation, as a Prolog program, and as RuleTaker triples
show_trace('young(bob) & kind(bob)\nforall x. (young(x) -> smart(x))\nforall x. (smart(x) & kind(x) -> successful(x))',
           'successful(bob)')
show_trace('young(bob). kind(bob).\nsmart(X) :- young(X).\nsuccessful(X) :- smart(X), kind(X).', '?- successful(bob).')
show_trace('("Bob" "is" "young" "+")\n("Bob" "is" "kind" "+")\n'
           '((("someone" "is" "young" "+")) -> ("someone" "is" "smart" "+"))\n'
           '((("someone" "is" "smart" "+") ("someone" "is" "kind" "+")) -> ("someone" "is" "successful" "+"))',
           '("Bob" "is" "successful" "+")')
# negation as failure, a first-order argument that needs resolution, and a counterfactual
show_trace('young(bob).\nrough(X) :- young(X), \\+ kind(X).', 'rough(bob)')
show_trace(['∀x (Human(x) → Eat(x))', '∀x (Eat(x) → ¬Plant(x))', '∀x (Plant(x) ⊕ Animal(x))',
            '∀x (Goat(x) → Animal(x))', '∀x (Dirt(x) → ¬Animal(x))', 'Goat(hulu) ∨ Human(hulu)'],
           'Animal(hulu) ∨ Dirt(hulu)')
_ = show_trace('We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. '
               'Blaf~Bern(0.2). We observed Trune.', 'Would Lumbo occur if not Ziklo instead of Ziklo?')

KNOWLEDGE BASE:
    Young(Bob)
    Kind(Bob)
    ∀x (Young(x) → Smart(x))
    ∀x (Smart(x) ∧ Kind(x) → Successful(x))
QUERY   : Successful(Bob)
INPUT SYNTAX: fol
METHOD  : forward chaining (closed world, stratified negation as failure)
FACTS   :
     young(bob)
     kind(bob)
RULES   :
     young(X) → smart(X)
     smart(X) ∧ kind(X) → successful(X)
QUERY   : successful(bob)
PROOF   :
     successful(bob)  [rule 2: smart(X) ∧ kind(X) → successful(X)]
       smart(bob)  [rule 1: young(X) → smart(X)]
         young(bob)  [given fact]
       kind(bob)  [given fact]
ANSWER  : True 
----------------------------------------------------------------------------------------------------
KNOWLEDGE BASE:
    young(bob) & kind(bob)
    forall x. (young(x) -> smart(x))
    forall x. (smart(x) & kind(x) -> successful(x))
QUERY   : successful(bob)
INPUT SYNTAX: fol
METHOD  : forward chaining (closed world, stratified negation as failure)
FACTS   :
     young(bob)
     kind(bob)
RULES   :
     young(X)

## 9. Load the public datasets

* **RuleTaker** — `tasksource/ruletaker` (test split; `entailment` → True, `not entailment` → False; `config` gives the
  depth). Translated into logic as described in section 6.
* **CounterBench** — `CounterBench/CounterBench` (`data_balanced_alpha_V1.json` + `data_balanced_backdoor_V2.json`;
  `yes` → True, `no` → False).
* **FOLIO** — `tasksource/folio` (validation split). The premises and conclusion are taken from the expert
  first-order annotations `premises-FOL` / `conclusion-FOL`, i.e. logic given directly. `True` → True,
  `False` / `Uncertain` → False.

A local copy in `data/` (e.g. `data/ruletaker_test.jsonl`, `data/counterbench.json`, `data/folio_validation.jsonl`)
is used instead of downloading when present.

In [10]:
import csv
import json
import os
import time
import urllib.request


def to_bool(value):
    """Map dataset labels to True (entailed) / False (not entailed)."""
    s = str(value).strip().lower()
    return s in {'true', '1', 'yes', 'y', 't', 'entailment', 'entailed', 'proved'}


def pick(record, names, default=None):
    for n in names:
        if n in record and record[n] not in (None, ''):
            return record[n]
    return default


# logic fields are preferred over natural-language fields when a record has both
CONTEXT_KEYS = ['premises-FOL', 'premises_fol', 'context_fol', 'fol_context', 'logic_context', 'context_logic',
                'kb', 'knowledge_base', 'context', 'premises', 'given_info', 'theory', 'facts', 'passage']
STATEMENT_KEYS = ['conclusion-FOL', 'conclusion_fol', 'statement_fol', 'fol_statement', 'query_fol',
                  'logic_statement', 'statement_logic', 'statement', 'query', 'question', 'conclusion',
                  'hypothesis', 'text', 'assertion']
LABEL_KEYS = ['flag', 'label', 'answer', 'gold', 'target']
ID_KEYS = ['id', 'example_id', 'question_id', 'qid', 'uid']


def normalise(record, prefix, index, group=None):
    context = pick(record, CONTEXT_KEYS, '')
    statement = pick(record, STATEMENT_KEYS, '')
    label = pick(record, LABEL_KEYS)
    rid = pick(record, ID_KEYS)
    return {'id': '%s-%s' % (prefix, rid if rid is not None else index), 'context': context,
            'statement': statement, 'label': None if label is None else to_bool(label),
            'group': group if group is not None else pick(record, ['config', 'type', 'depth'], prefix),
            'text': (pick(record, ['context', 'premises', 'given_info'], ''),
                     pick(record, ['statement', 'question', 'conclusion'], ''))}


def read_table(path):
    """Read a .csv / .json / .jsonl file into a list of dicts."""
    ext = os.path.splitext(path)[1].lower()
    with open(path, encoding='utf-8') as fh:
        if ext == '.csv':
            return list(csv.DictReader(fh))
        text = fh.read()
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            for k in ('data', 'examples', 'rows', 'test', 'validation', 'train'):
                if isinstance(data.get(k), list):
                    return data[k]
            return list(data.values()) if all(isinstance(v, dict) for v in data.values()) else [data]
        return data
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]


def local_file(name):
    for ext in ('.jsonl', '.json', '.csv'):
        p = os.path.join(DATA_DIR, name + ext)
        if os.path.exists(p):
            return p
    return None


def english_rules_to_logic(examples):
    """Data preparation for RuleTaker: the Hugging Face release is English only, so each theory and statement is
    translated into first-order logic once (the private test set provides logic directly)."""
    cache, converted = {}, 0
    for ex in examples:
        ctx = ex['context']
        if ctx not in cache:
            try:
                cache[ctx] = theory_to_fol(ctx)
            except Exception:
                cache[ctx] = (None, None)
        fol, names = cache[ctx]
        if fol is None:
            continue
        try:
            stmt = statement_to_fol(ex['statement'], names)
        except Exception:
            continue
        ex['context'], ex['statement'] = fol, stmt
        converted += 1
    return converted


def load_ruletaker():
    path = local_file('ruletaker_' + RULETAKER_SPLIT)
    if path:
        rows = read_table(path)
    else:
        from datasets import load_dataset
        rows = load_dataset('tasksource/ruletaker', split=RULETAKER_SPLIT)
    out = [normalise(dict(r), 'ruletaker', i) for i, r in enumerate(rows)]
    out = out[:MAX_PER_DATASET] if MAX_PER_DATASET else out
    n = english_rules_to_logic(out)
    print('  RuleTaker: %d / %d examples translated from English into logic' % (n, len(out)))
    return out


def load_counterbench():
    files = ['data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json']
    rows = []
    path = local_file('counterbench')
    if path:
        rows = read_table(path)
    else:
        from huggingface_hub import hf_hub_download
        for fn in files:
            p = hf_hub_download('CounterBench/CounterBench', fn, repo_type='dataset')
            part = read_table(p)
            tag = 'alpha' if 'alpha' in fn else 'backdoor'
            for r in part:
                r = dict(r)
                r['question_id'] = '%s-%s' % (tag, r.get('question_id'))
                rows.append(r)
    out = [normalise(dict(r), 'counterbench', i) for i, r in enumerate(rows)]
    return out[:MAX_PER_DATASET] if MAX_PER_DATASET else out


FOLIO_GITHUB = 'https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-%s.jsonl'


def load_folio():
    path = local_file('folio_' + FOLIO_SPLIT)
    if path:
        rows = read_table(path)
    else:
        try:
            from datasets import load_dataset
            rows = list(load_dataset('tasksource/folio', split=FOLIO_SPLIT))
        except Exception as exc:  # the same data from the FOLIO GitHub release
            print('  tasksource/folio unavailable (%s); using the FOLIO GitHub release' % type(exc).__name__)
            with urllib.request.urlopen(FOLIO_GITHUB % FOLIO_SPLIT) as resp:
                rows = [json.loads(l) for l in resp.read().decode('utf-8').splitlines() if l.strip()]
    rows = [r for r in rows if pick(dict(r), ['conclusion-FOL'])]     # keep examples that carry their logic
    out = [normalise(dict(r), 'folio', i) for i, r in enumerate(rows)]
    return out[:MAX_PER_DATASET] if MAX_PER_DATASET else out


DATASETS = {}
for name, loader in [('RuleTaker', load_ruletaker), ('CounterBench', load_counterbench), ('FOLIO', load_folio)]:
    try:
        t0 = time.perf_counter()
        DATASETS[name] = loader()
        print('%-12s %7d examples loaded (%.1fs)' % (name, len(DATASETS[name]), time.perf_counter() - t0))
    except Exception as exc:
        print('%-12s could not be loaded: %s: %s' % (name, type(exc).__name__, exc))
        print('             (pip install datasets huggingface_hub, check internet access, or put a copy in %s/)'
              % DATA_DIR)

for name, examples in DATASETS.items():
    if examples:
        ex = examples[0]
        ctx = ex['context'] if isinstance(ex['context'], str) else '\n'.join(map(str, ex['context']))
        print('\n[%s] first example as given to the reasoner:\n  context  : %s\n  statement: %s\n  label    : %s'
              % (name, ctx.replace('\n', ' | ')[:400], ex['statement'], ex['label']))

  RuleTaker: 151911 / 151911 examples translated from English into logic
RuleTaker     151911 examples loaded (15.3s)


CounterBench    1200 examples loaded (0.9s)


FOLIO            203 examples loaded (5.1s)

[RuleTaker] first example as given to the reasoner:
  context  : Big(anne) | Cold(anne) | Furry(anne) | Green(anne) | Red(anne) | Rough(anne) | Cold(erin) | Furry(erin) | Green(erin) | Round(erin) | ∀x (Cold(x) → Furry(x)) | ∀x (Round(x) → Rough(x)) | ∀x (Green(x) ∧ Cold(x) → Furry(x)) | ∀x (Furry(x) → Round(x)) | ∀x (Red(x) → Big(x)) | ∀x (Rough(x) → Furry(x)) | ∀x (Rough(x) → Cold(x)) | ∀x (Red(x) → Round(x))
  statement: Red(anne)
  label    : True

[CounterBench] first example as given to the reasoner:
  context  : We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.
  statement: Would Lumbo occur if not Ziklo instead of Ziklo?
  label    : False

[FOLIO] first example as given to the reasoner:
  context  : ∀x (InThisClub(x) ∧ PerformOftenIn(x, schoolTalentShow) → Attend(x, schoolEvent) ∧ VeryEngagedWith(x, schoolEvent)) | ∀x (InThisClub(x) → PerformOftenIn(x, schoolTalentShow) ⊕ (InActive(x) ∧ Dis

## 10. Evaluation: accuracy and inference time

Inference time covers parsing the logic and reasoning for every query (data loading excluded), measured with
`time.perf_counter`.

In [11]:
def evaluate(agent, examples):
    predictions = []
    t0 = time.perf_counter()
    for ex in examples:
        predictions.append(agent.predict(ex['context'], ex['statement']))
    elapsed = time.perf_counter() - t0
    by_group = defaultdict(lambda: [0, 0])
    correct = total = 0
    for p, ex in zip(predictions, examples):
        if ex['label'] is None:
            continue
        total += 1
        correct += int(p == ex['label'])
        by_group[str(ex['group'])][0] += int(p == ex['label'])
        by_group[str(ex['group'])][1] += 1
    return {'predictions': predictions, 'correct': correct, 'total': total, 'time': elapsed,
            'accuracy': 100.0 * correct / total if total else float('nan'), 'by_group': dict(by_group)}


agent = ReasoningAgent(SEMANTICS, PROVER_MAX_STEPS, PROVER_TIME_LIMIT)
RESULTS = {}
for name, examples in DATASETS.items():
    RESULTS[name] = evaluate(agent, examples)
    r = RESULTS[name]
    print('%-12s accuracy = %6.2f%%  (%d / %d)   inference time = %8.2f s   (%.3f ms / query)'
          % (name, r['accuracy'], r['correct'], r['total'], r['time'], 1000 * r['time'] / max(1, len(examples))))
    if len(r['by_group']) > 1:
        for g, (c, n) in sorted(r['by_group'].items()):
            print('      %-26s %6.2f%%  (%d / %d)' % (g, 100.0 * c / n, c, n))
print('\nInput routing  :', agent.stats)
print('Algorithm used :', agent.logic.stats)

RuleTaker    accuracy =  91.73%  (139341 / 151911)   inference time =    43.89 s   (0.289 ms / query)
      NatLang                    100.00%  (7793 / 7793)
      depth-0                     93.93%  (16294 / 17347)
      depth-1                     92.09%  (16381 / 17788)
      depth-2                     90.94%  (16134 / 17741)
      depth-3                     88.51%  (15915 / 17982)
      depth-3ext                  89.23%  (20720 / 23222)
      depth-3ext-NatLang          91.95%  (28519 / 31017)
      depth-5                     92.45%  (17585 / 19021)


CounterBench accuracy =  87.92%  (1055 / 1200)   inference time =     0.27 s   (0.223 ms / query)
      backdoor                    80.00%  (160 / 200)
      basic                      100.00%  (250 / 250)
      conditional                 66.00%  (165 / 250)
      joint                       96.00%  (240 / 250)
      nested                      96.00%  (240 / 250)


FOLIO        accuracy =  83.74%  (170 / 203)   inference time =     2.62 s   (12.903 ms / query)

Input routing  : {'causal model': 975, 'logic input': 152111, 'English rule theory -> logic': 0, 'unparsed': 228}
Algorithm used : {'forward-chaining (closed world)': 110546, 'forward-chaining (open world)': 7547, 'resolution': 34018}


## 11. Additional experiments

**(a) Semantics.** Each dataset is re-run with closed-world, open-world and automatic semantics, to show why the choice
matters and what the automatic rule costs or gains.

**(b) Notation independence.** The same RuleTaker knowledge bases are rewritten in two other logic notations (Prolog
and RuleTaker triples) and evaluated again; a general system should give the same answers.

In [12]:
for name, examples in DATASETS.items():
    sample = examples[:20000]
    row = []
    for sem in ('closed', 'open', 'auto'):
        r = evaluate(ReasoningAgent(sem, PROVER_MAX_STEPS, PROVER_TIME_LIMIT), sample)
        row.append('%s %6.2f%% (%.2fs)' % (sem, r['accuracy'], r['time']))
    print('%-12s n=%-6d  ' % (name, len(sample)) + '   '.join(row))


def _literal_in(fmt, lit):
    neg = lit.startswith('¬')
    m = re.match(r'(\w+)\(([^)]*)\)', lit.lstrip('¬'))
    pred, args = m.group(1), [a.strip() for a in m.group(2).split(',')]
    if fmt == 'prolog':
        s = '%s(%s)' % (pred.lower(), ', '.join('X' if a == 'x' else a.lower() for a in args))
        return ('\\+ ' if neg else '') + s
    words = ['something' if a == 'x' else a.replace('_', ' ') for a in args]
    if len(args) == 1:
        return '("%s" "is" "%s" "%s")' % (words[0], pred.lower(), '-' if neg else '+')
    return '("%s" "%s" "%s" "%s")' % (words[0], pred.lower(), words[1], '-' if neg else '+')


def rewrite(fol_lines, fmt):
    """Rewrite a translated RuleTaker theory (FOL strings) as a Prolog program or as RuleTaker triples."""
    out = []
    for line in fol_lines:
        m = re.match(r'^(?:∀x )?\((.*) → (.*)\)$', line)
        if m:
            body = [b.strip() for b in m.group(1).split('∧')]
            head = m.group(2).strip()
            if fmt == 'prolog':
                if head.startswith('¬'):
                    return None          # Prolog has no negative conclusions
                out.append('%s :- %s.' % (_literal_in(fmt, head), ', '.join(_literal_in(fmt, b) for b in body)))
            else:
                out.append('((%s) -> %s)' % (' '.join(_literal_in(fmt, b) for b in body), _literal_in(fmt, head)))
        elif fmt == 'prolog':
            if not line.startswith('¬'):
                out.append(_literal_in(fmt, line) + '.')
        else:
            out.append(_literal_in(fmt, line))
    return '\n'.join(out)


if 'RuleTaker' in DATASETS:
    sample = [ex for ex in DATASETS['RuleTaker'][:5000] if isinstance(ex['context'], list)]
    for fmt in ('fol', 'prolog', 'triples'):
        items = []
        for ex in sample:
            ctx = ex['context'] if fmt == 'fol' else rewrite(ex['context'], fmt)
            if ctx is None:
                continue
            q = ex['statement'] if fmt == 'fol' else (', ' if fmt == 'prolog' else ' ').join(
                _literal_in(fmt, l.strip()) for l in ex['statement'].split('∧'))
            items.append(dict(ex, context=ctx, statement=q))
        r = evaluate(ReasoningAgent(SEMANTICS, PROVER_MAX_STEPS, PROVER_TIME_LIMIT), items)
        print('RuleTaker written as %-8s accuracy = %6.2f%%  (%d queries, %.2f s)'
              % (fmt, r['accuracy'], r['total'], r['time']))
        if items and fmt != 'fol':
            print('      e.g. %s' % items[0]['context'].splitlines()[-1])

RuleTaker    n=20000   closed 100.00% (3.28s)   open  74.80% (9.23s)   auto  92.12% (5.35s)


CounterBench n=1200    closed  87.92% (0.26s)   open  87.92% (0.25s)   auto  87.92% (0.25s)


FOLIO        n=203     closed  84.24% (2.58s)   open  83.74% (2.66s)   auto  83.74% (2.79s)


RuleTaker written as fol      accuracy =  92.40%  (5000 queries, 1.24 s)


RuleTaker written as prolog   accuracy =  99.81%  (3636 queries, 0.47 s)
      e.g. round(X) :- red(X).


RuleTaker written as triples  accuracy = 100.00%  (5000 queries, 0.55 s)
      e.g. ((("something" "is" "red" "+")) -> ("something" "is" "round" "+"))


## 12. Reasoning traces on public examples

For each dataset, the correctly answered entailed query with the longest proof among the first 2,000 examples.

In [13]:
for name, examples in DATASETS.items():
    best, best_len = None, -1
    for ex, p in list(zip(examples, RESULTS[name]['predictions']))[:2000]:
        if p and ex['label']:
            tr = {}
            ReasoningAgent(SEMANTICS, PROVER_MAX_STEPS, PROVER_TIME_LIMIT).predict(ex['context'], ex['statement'], tr)
            n = len(tr.get('proof') or []) + len(tr.get('equations') or [])
            if n > best_len:
                best, best_len = ex, n
    if best is not None:
        print('[%s]  id = %s' % (name, best['id']))
        show_trace(best['context'], best['statement'], label=best['label'])

[RuleTaker]  id = ruletaker-842
KNOWLEDGE BASE:
    Cold(anne)
    Cold(dave)
    Furry(erin)
    Rough(harry)
    ∀x (Round(x) → ¬Young(x))
    ∀x (Young(x) ∧ Rough(x) → Quiet(x))
    ∀x (¬Young(x) → Quiet(x))
    ∀x (Red(x) ∧ Young(x) → Rough(x))
    ∀x (Cold(x) → Red(x))
    (¬Young(erin) → Quiet(erin))
QUERY   : Quiet(anne)
INPUT SYNTAX: fol
METHOD  : resolution refutation (set of support, uniform-cost clause selection)
PREMISES:
     cold(anne)
     cold(dave)
     furry(erin)
     rough(harry)
     ∀x (round(x) → ¬young(x))
     ∀x ((young(x) ∧ rough(x)) → quiet(x))
     ∀x (¬young(x) → quiet(x))
     ∀x ((red(x) ∧ young(x)) → rough(x))
     ∀x (cold(x) → red(x))
     (¬young(erin) → quiet(erin))
QUERY   : quiet(anne)
PROOF   :
      1. ¬quiet(anne)                                                 [negated query]
      2. quiet(v265912) ∨ young(v265912)                              [premise]
      3. young(anne)                                                  [resolve(1, 2)]
    

[FOLIO]  id = folio-1349
KNOWLEDGE BASE:
    ∀x (PetersPet(x) ∧ Mammal(x) → Animal(x))
    ∀x (Monkey(x) → Mammal(x))
    ∀x (PetersPet(x) → Monkey(x) ⊕  Bird(x))
    ∀x (Bird(x) → CanFly(x))
    ∀x (PetersPet(x) ∧ Animal(x) → Breathe(x))
    ∀x (PetersPet(x) ∧ CanFly(x) → ∃y ∃z (¬(y=z) ∧ Wing(y) ∧ Wing(z) ∧ Have(x, y) ∧ Have(x, z)))
    PeterSPet(rock)
    CanFly(rock) ∨ Bird(rock) ∨ ¬Breathe(rock)
QUERY   : ∃y ∃z (¬(y=z) ∧ Wing(y) ∧ Wing(z) ∧ Have(rock, y) ∧ Have(rock, z))
INPUT SYNTAX: fol
METHOD  : resolution refutation (set of support, uniform-cost clause selection)
PREMISES:
     ∀x ((peterspet(x) ∧ mammal(x)) → animal(x))
     ∀x (monkey(x) → mammal(x))
     ∀x (peterspet(x) → (monkey(x) ⊕ bird(x)))
     ∀x (bird(x) → canfly(x))
     ∀x ((peterspet(x) ∧ animal(x)) → breathe(x))
     ∀x ((peterspet(x) ∧ canfly(x)) → ∃y ∃z (¬=(y,z) ∧ wing(y) ∧ wing(z) ∧ have(x,y) ∧ have(x,z)))
     peterspet(rock)
     (canfly(rock) ∨ bird(rock) ∨ ¬breathe(rock))
QUERY   : ∃y ∃z (¬=(y,z) ∧ wing(y)

## 13. Predictions → `submission.csv`

If `TEST_FILE` is set, predictions are made for that file (columns `id`, `context`, `statement`; logic fields such as
`premises-FOL` / `conclusion-FOL` are also recognised). Otherwise the predictions for the public evaluation sets above
are written. Output columns: `id,label` with values `True` / `False`.

In [14]:
def predict_file(path, agent):
    rows = read_table(path)
    out = []
    t0 = time.perf_counter()
    for i, r in enumerate(rows):
        r = dict(r)
        rid = pick(r, ID_KEYS, i)
        out.append((rid, agent.predict(pick(r, CONTEXT_KEYS, ''), pick(r, STATEMENT_KEYS, ''))))
    return out, time.perf_counter() - t0


if TEST_FILE and os.path.exists(TEST_FILE):
    rows_out, test_time = predict_file(TEST_FILE, ReasoningAgent(SEMANTICS, PROVER_MAX_STEPS, PROVER_TIME_LIMIT))
    print('Predicted %d queries from %s in %.2f s' % (len(rows_out), TEST_FILE, test_time))
else:
    if TEST_FILE:
        print('TEST_FILE %r not found - writing predictions for the public evaluation sets instead' % TEST_FILE)
    rows_out = [(ex['id'], p) for name, examples in DATASETS.items()
                for ex, p in zip(examples, RESULTS[name]['predictions'])]
with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh)
    w.writerow(['id', LABEL_COLUMN])
    for rid, p in rows_out:
        w.writerow([rid, 'True' if p else 'False'])
print('Wrote %d predictions to %s' % (len(rows_out), os.path.abspath(OUTPUT_CSV)))

Wrote 153314 predictions to /home/priyesh/AI/submission/submission.csv


## 14. Final results

In [15]:
total_correct = sum(r['correct'] for r in RESULTS.values())
total_queries = sum(r['total'] for r in RESULTS.values())
total_time = sum(r['time'] for r in RESULTS.values())
print('=' * 78)
print('%-14s %12s %14s %18s' % ('Dataset', 'Accuracy', 'Queries', 'Inference time'))
print('-' * 78)
for name, r in RESULTS.items():
    print('%-14s %11.2f%% %14d %16.2f s' % (name, r['accuracy'], r['total'], r['time']))
print('-' * 78)
if total_queries:
    macro = sum(r['accuracy'] for r in RESULTS.values()) / len(RESULTS)
    print('%-14s %11.2f%% %14d %16.2f s' % ('ALL (micro)', 100.0 * total_correct / total_queries, total_queries, total_time))
    print('%-14s %11.2f%%' % ('ALL (macro)', macro))
print('=' * 78)

Dataset            Accuracy        Queries     Inference time
------------------------------------------------------------------------------
RuleTaker            91.73%         151911            43.89 s
CounterBench         87.92%           1200             0.27 s
FOLIO                83.74%            203             2.62 s
------------------------------------------------------------------------------
ALL (micro)          91.69%         153314            46.78 s
ALL (macro)          87.80%
